# Spatial Architecture of Kidney Injury: From Protein Fields to Tissue Logic

## A Note on This Notebook

> **Methodological note (post-hoc reframing).** This notebook was originally designed as exploratory analysis of kidney injury spatial organization. The candidate hypothesis it now synthesizes — *stromal-marker-positive tissue area appears less stromal-only and more multi-lineage at Sham→D7* — emerged post-hoc during Gate 6 normalization sensitivity (April 2026). The analyses in Parts 1–7 were designed BEFORE this hypothesis existed. This notebook synthesizes convergent post-hoc readings of pre-existing analyses; **it does not test the hypothesis.** Testing requires an n≥10 follow-up cohort. Pre-registration: `analysis_plans/temporal_interfaces_plan.md`. Throughline: `analysis_plans/kidney_notebook_throughline.md`.

## The Synthesis Question

**At Sham→D7, does stromal-marker-positive tissue area *appear* less stromal-only and more multi-lineage?**

The "appear" is load-bearing — we have no lineage tracing, no object-level transition analysis, and Family A's CLR transform mathematically couples some of the apparent co-movements. The notebook walks pre-existing analyses to ask whether multiple convergent readings are consistent with this candidate finding.

**CLR compositional constraint.** Family A reports both a stromal-only fraction *decrease* and a triple-positive interface fraction *increase*. On the closed simplex of 8 interface categories these are mathematically coupled — they are one degree of freedom dressed as two observations. Independent corroboration must come from non-CLR measures: Family C (raw-marker compartments, different markers and different math) and Part 2.5 (spatial coherence join-counts on binary indicators).

**Alternative hypothesis to discriminate.** The apparent shift could be a per-ROI sigmoid normalization artifact rather than biology. Gate 6 sensitivity showed the per-ROI normalization can manufacture compositional shifts (the `endothelial_clr` Sham→D7 finding collapses from g=+2.28 to g=+0.19 between regimes). The pilot data cannot discriminate redistribution-as-biology from redistribution-as-artifact. A follow-up cohort using Sham-reference normalization in the cell-type annotation engine — not per-ROI sigmoid — would resolve this.

## Reading Guide: Evidence Tiers

The 7 Parts of this notebook are sorted by what each contributes to the synthesis:

- **Tier 1 — direct candidate evidence.** Part 2 (Family A CLR co-headlines), Part 6 (Family C compartment trajectories).
- **Tier 2 — compatibility checks.** Part 2.5 (spatial coherence; non-compositional), Part 4 (Leiden cluster concordance with interface categories; same data through a different algorithm, NOT independent validation).
- **Tier 3 — context and caveats.** Part 1 (raw protein fields, descriptive), Part 3 (marker correlations, descriptive), Part 5 (neutrophil paradox as **negative control** for the candidate finding), Part 7 (legacy scale-dependent cluster counts; explicitly NOT used as scale-invariance evidence per pre-registration §2).

Tier 3 sections do not bear confirmatory weight on the candidate hypothesis. They provide context, caveats, and the negative control.

---

## The UUO Injury Model: Mechanical Cascade

**Unilateral Ureteral Obstruction (UUO)** is a surgical model of progressive kidney fibrosis:

1. **Ureteral ligation** → urinary backflow → hydronephrosis (kidney swelling)
2. **Pressure buildup** → tubular dilation → rising interstitial pressure
3. **Mechanical stress** → inflammation trigger
4. **Secondary damage**: Inflammation → vascular rarefaction → hypoxia → fibrosis

By Day 7, tissue is on a trajectory whose endpoint (resolution vs fibrotic remodeling) the pilot cannot itself discriminate. The pre-registered Family C compartment-activation trajectories quantify D7-state shifts as candidate signals for an n≥10 follow-up; the pilot does not characterize fate at n=2.

### Timeline (Expected Biology)

- **Day 1**: Acute inflammation - neutrophil recruitment (Ly6G↑), focal infiltration
- **Day 3**: Peak inflammation - macrophage activation (CD11b↑, CD206↑), expanding response
- **Day 7**: Resolution or fibrosis - persistent immune cells, fibroblast activation (CD140a/b↑, CD44↑)

### Kidney Anatomy: Two Worlds

**Cortex** (outer kidney):
- Glomerular filtering apparatus (~200μm architectural units)
- Dense CD31+/CD34+ vascular networks
- Better perfusion → better immune access
- Expected: vascular enrichment, organized glomerular structures

**Medulla** (inner kidney):
- Concentrating tubules (~75μm cross-sections)
- Sparser CD31+/CD34+ vasculature
- Hypoxic baseline, slower immune access
- Expected: tubular dominance, perivascular inflammation


In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import gzip
import json
from scipy.spatial.distance import cdist
from scipy import stats

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 10

def deserialize_array(arr_dict):
    """Convert stored numpy array back to numpy array"""
    if isinstance(arr_dict, dict) and '__numpy_array__' in arr_dict:
        data = arr_dict['data']
        dtype = arr_dict['dtype']
        shape = arr_dict['shape']
        return np.array(data, dtype=dtype).reshape(shape)
    return arr_dict

# --- DNA reference underlay support ---
_raw_data_dir = Path('../../data/241218_IMC_Alun')
if not _raw_data_dir.exists():
    _raw_data_dir = Path('data/241218_IMC_Alun')
    if not _raw_data_dir.exists():
        _raw_data_dir = Path.home() / 'Documents' / 'IMC' / 'data' / '241218_IMC_Alun'

_dna_cache = {}

def load_dna_reference(roi_name: str):
    """Load DNA composite image and bounds for a given ROI name."""
    if roi_name in _dna_cache:
        return _dna_cache[roi_name]
    txt_file = _raw_data_dir / f'{roi_name}.txt'
    if not txt_file.exists():
        _dna_cache[roi_name] = (None, None)
        return None, None
    raw = pd.read_csv(txt_file, sep='\t')
    x, y = raw['X'].values.astype(int), raw['Y'].values.astype(int)
    dna_cols = [c for c in raw.columns if 'DNA' in c]
    dna1 = raw[dna_cols[0]].values
    dna2 = raw[dna_cols[1]].values if len(dna_cols) > 1 else dna1
    composite = (dna1 + dna2) / 2.0
    x_min, x_max = int(x.min()), int(x.max())
    y_min, y_max = int(y.min()), int(y.max())
    img = np.zeros((y_max - y_min + 1, x_max - x_min + 1), dtype=float)
    img[y - y_min, x - x_min] = composite
    img_pos = np.where(img > 0, img, 0)
    img_log = np.log1p(img_pos)
    p1, p99 = np.percentile(img_log[img_log > 0], [1, 99]) if (img_log > 0).any() else (0, 1)
    img_norm = np.clip((img_log - p1) / max(p99 - p1, 1e-10), 0, 1)
    bounds = (x_min, x_max, y_min, y_max)
    _dna_cache[roi_name] = (img_norm, bounds)
    return img_norm, bounds

def render_dna_underlay(ax, roi_name: str, alpha=0.25):
    """Render DNA tissue underlay on an axes. Returns bounds or None."""
    dna_img, bounds = load_dna_reference(roi_name)
    if dna_img is None:
        return None
    ax.imshow(dna_img, extent=list(bounds), origin='lower',
              cmap='gray', alpha=alpha, aspect='equal', zorder=0)
    return bounds

print("Setup complete")
print(f"  Raw data dir: {_raw_data_dir} ({'found' if _raw_data_dir.exists() else 'NOT FOUND'})")

# Project root for annotation and config paths
project_root = Path('../../')
if not (project_root / 'config.json').exists():
    project_root = Path.home() / 'Documents' / 'IMC'

---

# Part 1: Protein Fields Before Clustering

## The Abstraction Problem

Before we cluster superpixels or gate phenotypes, we face a fundamental question: **what do the raw protein measurements tell us?**

Each superpixel expresses 9 proteins at varying intensities. These are **continuous fields**, not discrete identities. A "macrophage" is not a binary state—it's a gradient of CD11b expression, CD206 abundance, CD45 presence. 

Let's examine the protein fields directly to understand what patterns exist before any computational abstraction.

In [ ]:
# Load all ROI results
results_dir = Path('../../results/roi_results')
if not results_dir.exists():
    results_dir = Path('results/roi_results')
    if not results_dir.exists():
        results_dir = Path.home() / 'Documents' / 'IMC' / 'results' / 'roi_results'
result_files = sorted(results_dir.glob('roi_*.json.gz'))

markers = ['CD45', 'CD11b', 'Ly6G', 'CD140a', 'CD140b', 'CD31', 'CD34', 'CD206', 'CD44']

all_superpixels = []

for rf in result_files:
    # Parse metadata from filename
    roi_name = rf.name.replace('_results.json.gz', '').replace('roi_', '')
    
    if 'Test01' in roi_name:
        continue
    
    # Extract timepoint and mouse
    if 'D1' in roi_name:
        timepoint = 'D1'
    elif 'D3' in roi_name:
        timepoint = 'D3'
    elif 'D7' in roi_name:
        timepoint = 'D7'
    else:
        continue
    
    mouse = 'M1' if '_M1_' in roi_name else 'M2'
    
    # Load result file
    with gzip.open(rf, 'rt') as f:
        result = json.load(f)
    
    # Process each scale
    for scale_name in ['10.0', '20.0', '40.0']:
        scale_data = result['multiscale_results'][scale_name]
        
        # Deserialize arrays
        cluster_labels = deserialize_array(scale_data['cluster_labels'])
        coords = deserialize_array(scale_data['superpixel_coords'])
        marker_data = {m: deserialize_array(scale_data['transformed_arrays'][m]) 
                      for m in markers}
        
        # Create superpixel dataframe
        n_spx = len(cluster_labels)
        for i in range(n_spx):
            row = {
                'roi': roi_name,
                'timepoint': timepoint,
                'mouse': mouse,
                'scale_um': float(scale_name),
                'cluster': int(cluster_labels[i]),
                'x': coords[i, 0],
                'y': coords[i, 1],
            }
            
            for m in markers:
                row[m] = marker_data[m][i]
            
            all_superpixels.append(row)

df = pd.DataFrame(all_superpixels)

print(f"Loaded {len(df)} superpixels")
print(f"  - {df['roi'].nunique()} ROIs")
print(f"  - Timepoints: {sorted(df['timepoint'].unique())}")
print(f"  - Mice: {sorted(df['mouse'].unique())}")
print(f"  - Scales: {sorted(df['scale_um'].unique())} μm")
print(f"  - Markers: {markers}")

In [ ]:
# Focus on 10μm scale for protein field analysis
df_10 = df[df['scale_um'] == 10.0].copy()

# Protein field distributions
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.ravel()

for idx, marker in enumerate(markers):
    ax = axes[idx]
    
    # Plot distribution for each timepoint
    for tp in ['Sham', 'D1', 'D3', 'D7']:
        tp_data = df_10[df_10['timepoint'] == tp][marker]
        ax.hist(tp_data, bins=50, alpha=0.5, label=tp, density=True)
    
    ax.set_xlabel(f'{marker} Expression (arcsinh)', fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.set_title(marker, fontweight='bold', fontsize=11)
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Protein Field Distributions Across Injury Timeline', 
             fontweight='bold', fontsize=14, y=0.995)
plt.tight_layout()
plt.show()

print(f"Loaded {len(df_10)} superpixels at 10μm scale")
print(f"  - {df_10['roi'].nunique()} ROIs")
print(f"  - Timepoints: {sorted(df_10['timepoint'].unique())}")
print(f"  - Markers: {markers}")

### Reading the Protein Fields

These distributions reveal the **continuous nature of protein expression** before we impose discrete categories.

**Immune markers** (CD45, CD11b, CD206):
- **Bimodal distributions** - two peaks corresponding to immune-positive and immune-negative tissue
- CD45 and CD11b show **rightward shift D1→D7** - progressive immune infiltration  
- CD206 (M2 marker) shows subtle increase - M2s present but not dominant

**Stromal markers** (CD140a, CD140b, CD44):
- **Broad unimodal distributions** - most tissue expresses these at intermediate levels
- CD44 shows **dramatic rightward shift at D7** - late-stage activation increase
- CD140a/CD140b relatively stable - constitutive fibroblast/pericyte presence

**Vascular markers** (CD31, CD34):
- **Narrow distributions** - structurally defined (vessels are either present or absent)
- Minimal temporal shift - vascular architecture stable (though cells may be depleted, vessel locations don't change)

**Key insight**: The protein fields are NOT cleanly separated. There's no clear "immune" vs "non-immune" boundary—it's a continuum. Boolean gating will draw sharp lines through these gradients. Clustering will find groupings in this high-dimensional space. Both are **abstractions** of the underlying continuous biology.

---

**Part 1 framing (Tier 3 context).** These descriptive observations frame what follows. Stromal markers (CD140a, CD140b) are stable across timepoints in raw expression, yet *compositional classification* may still shift — Part 2 quantifies that classification at the lineage level. Part 1 is Tier 3 context: it does not bear confirmatory weight on the candidate finding.


---

# Part 2: Two Lenses on Identity — Discrete Gates and Continuous Memberships

## The Annotation Challenge

Each superpixel expresses 9 proteins at varying intensities. To make biological sense of this, we need to assign identities. But **how**?

We use two complementary systems, each capturing something the other misses:

**1. Discrete Boolean Gating** — 15 cell types via marker positivity rules (CD45+/CD11b+/CD206+ = M2 macrophage). This gives interpretable phenotype labels that connect to immunology conventions. But with only 9 markers, **~76% of superpixels remain unassigned** — the panel cannot name them.

**2. Continuous Multi-Label Memberships** — three independent axes (lineage / subtype / activation) scored [0,1] per superpixel. Crucially, lineage scores are **non-exclusive**: a superpixel at a vessel-stroma boundary scores on both endothelial and stromal lineages simultaneously. This recovers the ~48% of tissue that boolean gating discards as "unassigned."

Both annotation modes are computed by `batch_annotate_all_rois.py`, which calls the annotation engine (`src/analysis/cell_type_annotation.py`) on each ROI's pipeline output. Per-ROI Parquet files contain discrete labels, continuous lineage/subtype/activation scores, and spatial coordinates.

Let's load the annotations and see what each lens reveals.

For a detailed quantification of information loss from boolean discretization, see the companion methods notebook: `notebooks/methods_validation/01_technical_methods/gradient_discretization.ipynb`.

In [ ]:
# Load cell type annotations from pipeline output (15 discrete types + continuous memberships)
from pathlib import Path
import json

annotations_dir = Path(project_root) / 'results' / 'biological_analysis' / 'cell_type_annotations'
parquet_files = sorted(annotations_dir.glob('roi_*_cell_types.parquet'))

# Load all annotations with metadata
ann_frames = []
for pf in parquet_files:
    roi_id = pf.stem.replace('_cell_types', '')
    adf = pd.read_parquet(pf)
    adf['roi'] = roi_id
    # Extract timepoint from ROI ID
    if 'Sam' in roi_id:
        adf['timepoint'] = 'Sham'
    elif '_D1_' in roi_id:
        adf['timepoint'] = 'D1'
    elif '_D3_' in roi_id:
        adf['timepoint'] = 'D3'
    elif '_D7_' in roi_id:
        adf['timepoint'] = 'D7'
    else:
        adf['timepoint'] = 'Unknown'
    ann_frames.append(adf)

ann_df = pd.concat(ann_frames, ignore_index=True)
ann_df = ann_df[ann_df['timepoint'] != 'Unknown']

# Load cell type colors from config
with open(Path(project_root) / 'config.json') as f:
    config = json.load(f)
ct_defs = config['cell_type_annotation']['cell_types']
ct_colors = {name: defn.get('color', '#888888') for name, defn in ct_defs.items()}

# Summary
all_types = sorted([t for t in ann_df['cell_type'].unique() if t != 'unassigned'])
print(f"Loaded {len(ann_df)} superpixels from {ann_df['roi'].nunique()} ROIs")
print(f"Discrete cell types: {len(all_types)}")
print(f"Lineage columns: {[c for c in ann_df.columns if c.startswith('lineage_')]}")
print(f"Assignment rate: {(ann_df['cell_type'] != 'unassigned').mean():.1%}")

In [ ]:
# Temporal dynamics of all 15 cell types
timepoint_order = ['Sham', 'D1', 'D3', 'D7']

# Compute proportions per ROI (assigned-only denominator)
type_temporal = []
for roi in ann_df['roi'].unique():
    roi_data = ann_df[ann_df['roi'] == roi]
    tp = roi_data['timepoint'].iloc[0]
    n_assigned = (roi_data['cell_type'] != 'unassigned').sum()
    if n_assigned == 0:
        continue
    for ct in all_types:
        type_temporal.append({
            'roi': roi, 'timepoint': tp, 'cell_type': ct,
            'proportion': (roi_data['cell_type'] == ct).sum() / n_assigned * 100
        })

type_df = pd.DataFrame(type_temporal)
type_df['timepoint'] = pd.Categorical(type_df['timepoint'], categories=timepoint_order, ordered=True)

# Plot top 8 types by total count
top_types = ann_df[ann_df['cell_type'] != 'unassigned']['cell_type'].value_counts().head(8).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=False)
axes = axes.ravel()

for idx, ct in enumerate(top_types):
    ax = axes[idx]
    ct_data = type_df[type_df['cell_type'] == ct]
    color = ct_colors.get(ct, '#888888')
    
    for tp_idx, tp in enumerate(timepoint_order):
        vals = ct_data[ct_data['timepoint'] == tp]['proportion']
        ax.bar(tp_idx, vals.mean(), yerr=vals.std() if len(vals) > 1 else 0,
               color=color, capsize=3, alpha=0.8, edgecolor='white')
    
    ax.set_xticks(range(4))
    ax.set_xticklabels(timepoint_order, fontsize=8)
    ax.set_title(ct.replace('_', ' ').title(), fontsize=9, fontweight='bold')
    ax.set_ylabel('% Assigned' if idx % 4 == 0 else '')
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Temporal Dynamics of Cell Phenotypes (15-Type Gating)', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.show()

### The 76% Problem

The bar charts above show temporal dynamics for the ~24% of tissue that boolean gating can name. But **what about the rest?**

Most "unassigned" superpixels aren't empty — they express proteins, just not in combinations that match any gate. Many sit at **tissue interfaces** where lineages blend: a peritubular capillary wall where endothelial and stromal signals overlap, or an infiltration front where immune and stromal markers co-express.

The continuous membership system treats these interfaces as **first-class citizens**. Each superpixel gets independent scores on three axes:
- **Lineage** (immune / endothelial / stromal) — sigmoid-normalized, non-exclusive
- **Subtype** (within-lineage: neutrophil, M2, myeloid, non-myeloid) — evaluated only when parent lineage exceeds threshold
- **Activation** (CD44, CD140b) — independent overlay, applies regardless of lineage

Below, we visualize the lineage axis as a ternary RGB map: Red = Immune, Green = Stromal, Blue = Endothelial. Mixed colors reveal interfaces.

In [ ]:
# Ternary lineage maps: one representative ROI per timepoint
# RGB: Red=Immune, Green=Stromal, Blue=Endothelial

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for tp_idx, tp in enumerate(timepoint_order):
    ax = axes[tp_idx]
    tp_rois = ann_df[ann_df['timepoint'] == tp]['roi'].unique()
    if len(tp_rois) == 0:
        ax.axis('off')
        continue
    
    roi_id = sorted(tp_rois)[0]
    roi_data = ann_df[ann_df['roi'] == roi_id]
    
    # Build RGB from lineage scores
    r = roi_data['lineage_immune'].values
    g = roi_data['lineage_stromal'].values
    b = roi_data['lineage_endothelial'].values
    rgb = np.column_stack([r, g, b])
    rgb = np.clip(rgb, 0, 1)
    
    # Global contrast stretch (preserve relative intensity)
    row_max = rgb.max(axis=1)
    p5, p95 = np.percentile(row_max, [5, 95])
    brightness = np.clip((row_max - p5) / max(p95 - p5, 0.01), 0, 1) * 0.8 + 0.2
    current_max = np.maximum(row_max, 1e-6)
    rgb_display = rgb * (brightness / current_max)[:, np.newaxis]
    rgb_display = np.clip(rgb_display, 0, 1)
    rgb_display[row_max < 0.15] = 0.15  # dark gray for no-lineage
    
    ax.scatter(roi_data['x'], roi_data['y'], c=rgb_display, s=1.5, marker='s', edgecolors='none')
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.axis('off')
    ax.set_title(f'{tp}', fontsize=12, fontweight='bold')

# Legend
from matplotlib.patches import Patch
legend_items = [
    Patch(facecolor='#FF0000', label='Immune'),
    Patch(facecolor='#00FF00', label='Stromal'),
    Patch(facecolor='#0000FF', label='Endothelial'),
    Patch(facecolor='#FF00FF', label='Immune+Endothelial'),
    Patch(facecolor='#FFFF00', label='Immune+Stromal'),
    Patch(facecolor='#00FFFF', label='Endothelial+Stromal'),
    Patch(facecolor='#262626', label='No lineage'),
]
fig.legend(handles=legend_items, loc='lower center', ncol=7, fontsize=8,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('Ternary Lineage Landscape Across Injury Progression',
             fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

In [ ]:
# Mouse-level interface composition trajectories
# Replaces the prior pseudoreplicated stacked-bar (which pooled all superpixels per timepoint).
# Now consumes the pre-registered output from run_temporal_interface_analysis.py.
#
# Plan reference: analysis_plans/temporal_interfaces_plan.md §9 (cell 11 replacement spec).
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

interface_path = project_root / 'results' / 'biological_analysis' / 'temporal_interfaces' / 'interface_fractions.parquet'
fractions = pd.read_parquet(interface_path)
fractions = fractions.sort_values(['timepoint', 'mouse']).reset_index(drop=True)
timepoint_order = ['Sham', 'D1', 'D3', 'D7']

INTERFACE_CATEGORIES = [
    'immune', 'endothelial', 'stromal',
    'endothelial+immune', 'immune+stromal', 'endothelial+stromal',
    'endothelial+immune+stromal', 'none',
]
interface_colors = {
    'immune': '#E63946', 'endothelial': '#457B9D', 'stromal': '#8AB17D',
    'endothelial+immune': '#FF00FF', 'immune+stromal': '#CCCC00',
    'endothelial+stromal': '#00CCCC', 'endothelial+immune+stromal': '#999999',
    'none': '#333333',
}

# Primary view: mouse-level dot plot. Two dots per timepoint per category (n=2 mice).
fig, axes = plt.subplots(2, 4, figsize=(15, 7), sharex=True)
axes = axes.flatten()
x_positions = {tp: i for i, tp in enumerate(timepoint_order)}
for ax, category in zip(axes, INTERFACE_CATEGORIES):
    color = interface_colors.get(category, '#888888')
    for tp in timepoint_order:
        sub = fractions[fractions['timepoint'] == tp]
        if category not in sub.columns or sub.empty:
            continue
        xs = np.full(len(sub), x_positions[tp]) + np.random.RandomState(42).uniform(-0.08, 0.08, size=len(sub))
        ax.scatter(xs, sub[category].values * 100, c=color, s=80, edgecolor='black', linewidth=0.6, zorder=3)
        ax.hlines(sub[category].mean() * 100, x_positions[tp]-0.18, x_positions[tp]+0.18, color=color, linewidth=2.5, alpha=0.6)
    ax.set_xticks(range(len(timepoint_order)))
    ax.set_xticklabels(timepoint_order, fontsize=10)
    ax.set_title(category.replace('+', ' + ').title(), fontsize=10, fontweight='bold')
    ax.set_ylabel('% of superpixels', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
fig.suptitle('Mouse-level interface composition (2 dots per timepoint = 2 mice; horizontal bar = mean)',
             fontsize=12, fontweight='bold', y=1.0)
fig.tight_layout()
plt.show()

# Secondary descriptive: stacked bar of mean fractions per timepoint
# Explicitly labeled as pooled view, NOT inferential
mean_pivot = (
    fractions.groupby('timepoint')[INTERFACE_CATEGORIES].mean()
    .reindex(timepoint_order)
)
fig2, ax2 = plt.subplots(figsize=(8, 4.5))
bottom = np.zeros(len(timepoint_order))
for cat in INTERFACE_CATEGORIES:
    if cat not in mean_pivot.columns:
        continue
    vals = mean_pivot[cat].values * 100
    ax2.bar(range(len(timepoint_order)), vals, bottom=bottom,
            label=cat.replace('+', ' + ').title(),
            color=interface_colors.get(cat, '#888888'), edgecolor='white', linewidth=0.4)
    bottom += vals
ax2.set_xticks(range(len(timepoint_order)))
ax2.set_xticklabels(timepoint_order, fontsize=11)
ax2.set_ylabel('% of superpixels (mouse-mean)')
ax2.set_title('Pooled descriptive view (NOT inferential — see dot plot above for variability)',
              fontsize=11, fontstyle='italic')
ax2.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
fig2.tight_layout()
plt.show()

# Quick numeric summary at the primary threshold
print(f"Loaded {len(fractions)} mouse-level rows ({fractions['mouse'].nunique()} mice across {fractions['timepoint'].nunique()} timepoints)")
print(f"Threshold used: {fractions['threshold'].iloc[0]} (sensitivity at 0.2/0.4 in sensitivity_thresholds.parquet)")
print(f"Total superpixels per mouse range: {int(fractions['n_total'].min())}–{int(fractions['n_total'].max())}")


### What the Interfaces Reveal (Quantified)

> **CLR compositional constraint (must read before the table below).** On the closed simplex of 8 interface categories, the stromal-only decrease and triple-positive increase reported below are mathematically coupled — they are one degree of freedom dressed as two observations. Family A alone is insufficient evidence for the candidate hypothesis. Independent corroboration requires non-CLR measures: Family C (Part 6) and the spatial coherence below (Part 2.5).



The mouse-level dot plot above visualizes per-mouse interface category fractions; the table below summarizes pre-registered effect sizes from `endpoint_summary.csv`. **This section's co-headline selection is a post-hoc reframing after Gate 6 exposed the per-ROI normalization confound; the reframing itself is not pre-registered but is constrained by the pre-registered sensitivity outputs.**

The Family A pre-registered analysis runs in TWO normalization regimes (per-ROI sigmoid vs Sham-reference raw-marker thresholds at the 75th-percentile of the Sham distribution, swept at {65, 75, 85} percentiles for sensitivity).

#### Full Family A Sham→D7 CLR landscape (all 8 endpoints, no selection)

| Endpoint | Per-ROI g | Sham-ref g | Sign reverse? | Magnitude collapse? | Included as co-headline? |
|---|---|---|---|---|---|
| Stromal-only | -1.59 | -3.20 | No | No | **Yes — direction-consistent; Sham-ref strengthens** |
| Endothelial+immune+stromal (triple-positive) | +0.63 | +2.08 | No | No | **Yes — direction-consistent; Sham-ref strengthens** |
| Immune-only | -0.53 | -0.49 | No | No | No (stable but small; near pilot noise floor) |
| Endothelial-only | +2.28 | +0.19 | No | **Yes** | No — demoted to methods finding (see below) |
| Endothelial+immune | -0.17 | **+2.20** | **Yes** | No | No — sign reverse disqualifies as robust co-headline |
| Immune+stromal | -0.03 | +1.23 | **Yes** | No | No — sign reverse; per-ROI near zero |
| Endothelial+stromal | -0.005 | -0.42 | No | No | No (near zero in both regimes) |
| None | +0.33 | -12.46 | **Yes** | No | No — sign reverse; `g_pathological` territory under Sham-ref |

**Selection rule for co-headline inclusion**: `(1)` direction-consistent between normalization regimes (no sign reverse in the Sham→D7 contrast), `(2)` per-ROI |g| > 0.5 (above pilot noise floor), `(3)` not magnitude-collapsing (Sham-ref magnitude at least 20% of per-ROI magnitude). Two of 8 endpoints meet all three criteria: stromal-only and triple-positive. The other 6 are either small in both regimes, sign-reverse, or magnitude-collapse. This is a post-hoc filter; adequately-powered follow-up should confirm whether these selection criteria identify the same endpoints.

#### Promoted co-headlines (D7-contrast normalization-consistent)

| Endpoint | Per-ROI g | Sham-ref g | Shrunk (skeptical / neutral / optimistic) | n required (skep / neut / opt) |
|---|---|---|---|---|
| Stromal-only (CLR) | -1.59 | -3.20 | -0.25 / -0.69 / -1.20 | 244 / 34 / 11 |
| Triple-positive interface (CLR) | +0.63 | +2.08 | +0.07 / +0.20 / +0.41 | many hundreds / ~150 / ~50 |

**Trajectory caveat (important)**: the "normalization-consistent" label applies only to the *Sham→D7 contrast*. Early-timepoint contrasts in these same endpoints DO sign-reverse between regimes (stromal-only Sham→D1: per-ROI +0.23 → Sham-ref -0.23; Sham→D3: per-ROI +0.18 → Sham-ref -1.69; triple-positive D1→D3: per-ROI -0.20 → Sham-ref +1.37). The *shape* of the temporal trajectory is normalization-dependent; only the *D7 endpoint* direction is consistent.

**Biological reading**: the stromal-only decrease and triple-positive increase at Sham→D7 are **consistent with a hypothesis** that stromal-marker-positive tissue area transitions from stromal-only classification to multi-lineage interface classification. This is *convergent evidence*, not a demonstrated mechanism — we have no lineage tracing, no object-level transition analysis, and the CLR compositional transform mechanically couples these two components. Family C's `triple_overlap_fraction` (g=+3.29) is *related but non-identical corroboration* — it uses CD31+CD140b compartment thresholds whereas Family A's stromal axis uses CD140a and endothelial axis uses mean(CD31, CD34). Convergent across measurement schemes, not replication of one construct.

#### Methods finding (do NOT carry forward as a biology headline)

| Endpoint | Per-ROI g | Sham-ref g | Status |
|---|---|---|---|
| Endothelial-only (CLR) | +2.28 | +0.19 | `normalization_g_collapse=True`. The per-ROI sigmoid normalization manufactures a ~12× inflation of a small compositional shift. D1→D7 and D3→D7 also sign-reverse between regimes. The per-ROI regime either inflates or reverses this endpoint depending on the contrast; we note this as evidence that normalization choice materially affects CLR conclusions — NOT as evidence Sham-ref is "correct" (we have not demonstrated that either regime is artifact-free). |

#### Required reading per pre-registration §10

> "Mouse-level mean shifted from X (range a-b) to Y (range c-d), Hedges' g = Z. Bayesian-shrunk under three priors (skeptical/neutral/optimistic): Z_sk / Z_ne / Z_op. Detecting these shrunk effects at 80% power would require n ≥ N_sk / N_ne / N_op mice per group respectively. Current pilot at n=2 cannot distinguish any of these from sampling variance; the three-prior range transparently exposes the uncertainty. Findings flagged `normalization_sign_reverse` or `normalization_g_collapse` between regimes carry an additional caveat: the magnitude itself is normalization-dependent."

#### Multiplicity, pathology, missingness

At n=2 per group no real p-value exists; no BH-FDR is computed (earlier drafts used a proxy column that Gate 6 removed). 8 endpoint rows have |g|>3 with pooled_std<0.01 (variance-collapse artifacts) → flagged `g_pathological=True`, NaN shrunk values, NaN n_required. 27 rows have insufficient_support (n<2 mice in one group) → preserved as NaN-with-flag rather than silently dropped. 2/11 endpoints with per-ROI |g|>0.5 sign-reverse between normalization regimes. Rank-ordering audit is available by sorting endpoint_summary.csv on `|hedges_g|`.

For the continuous neighborhood heatmap (Family B) and join-count spatial coherence, see the next section.

---

**Bridge to Part 2.5 (Tier 2 compatibility check).** If the candidate finding is more than CLR coupling, the spatial coherence of triple-positive interfaces should also strengthen — a non-compositional measure that, if it agrees with the Family A direction, provides independent corroboration. Part 2.5 tests this with k-NN neighbor-minus-self deltas and join-count statistics on binary interface indicators.


---

# Part 2.5: Continuous Neighborhood Lineage Shifts and Spatial Coherence

## What the Orphaned CSV Was Hiding

The original `spatial_neighborhood_analysis.py` self-stratified continuous neighborhood path produced an output (`continuous_neighborhood_summary.csv`, since deleted) that no notebook consumed. The replacement pipeline (`run_temporal_interface_analysis.py`) computes a corrected version with neighbor-minus-self framing, exposed via `continuous_neighborhood_temporal.parquet`. Two corrections from the pre-registration:

1. **Neighbor-minus-self framing**: the prior CSV stratified neighbor lineage scores by `composite_label`, but `composite_label` is itself derived from those same scores (tautological). We now report `neighbor_lineage_X − self_lineage_X` per (cell type × lineage). Within stratum X, this delta is biased negative for the self-matching lineage by construction — so **only temporal *changes* in the delta are interpretable**, not absolute values.

2. **Trajectory filter** (plan §3 Family B): a composite label must have ≥1 mouse with valid delta in *every* timepoint to be retained. Labels failing in any timepoint are excluded from the trajectory analysis, with the missingness signal preserved in `continuous_neighborhood_missingness.parquet`.

The plot below renders `delta_vs_Sham` (mouse-level mean delta minus the Sham-baseline delta for that label×lineage). Values are interpretable as "how does this cell type's neighborhood lineage composition shift relative to Sham at each timepoint?"

The companion plot shows join-count z-scores per interface category per ROI — a categorical-data spatial-autocorrelation statistic. High positive z indicates spatially clustered (not dispersed) interface zones.

**Family B caution (per plan):** 100% of Family B endpoints (252/252) are flagged threshold-sensitive across the min-support sweep {10, 20, 40}. This is structural — changing the support floor changes which composite labels survive the trajectory filter. Treat the heatmap as ranked exploration, not as discoveries.

**Tier label.** Part 2.5 is a **Tier 2 compatibility check** — non-compositional spatial measures on the same superpixels. If join-count statistics on binary triple-positive indicators show spatial clustering at D7 (and not at Sham), that's independent of Family A's CLR transform. If they don't, Family A's CLR signal lacks spatial coherence support.


In [ ]:
# Continuous neighborhood + spatial coherence visualization
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

base = project_root / 'results' / 'biological_analysis' / 'temporal_interfaces'
neigh = pd.read_parquet(base / 'continuous_neighborhood_temporal.parquet')
join_counts = pd.read_parquet(base / 'join_counts.parquet')
missingness = pd.read_parquet(base / 'continuous_neighborhood_missingness.parquet')

timepoint_order = ['Sham', 'D1', 'D3', 'D7']

# === Plot 1: heatmap of mouse-level delta-vs-Sham per (composite_label x neighbor_lineage), faceted by timepoint ===
delta_cols = ['vs_sham_mean_delta_lineage_immune', 'vs_sham_mean_delta_lineage_endothelial', 'vs_sham_mean_delta_lineage_stromal']
short_lin = ['immune', 'endothelial', 'stromal']

# Aggregate to mouse-mean per (label, timepoint)
agg = neigh.groupby(['composite_label', 'timepoint'])[delta_cols].mean().reset_index()
labels_kept = sorted(agg['composite_label'].unique())

fig, axes = plt.subplots(1, len(timepoint_order), figsize=(4.5 * len(timepoint_order), max(4, 0.35 * len(labels_kept))), sharey=True)
vmax = float(np.abs(agg[delta_cols]).max().max()) or 0.1
for ax, tp in zip(axes, timepoint_order):
    sub = agg[agg['timepoint'] == tp].set_index('composite_label').reindex(labels_kept)
    matrix = sub[delta_cols].values
    im = ax.imshow(matrix, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
    ax.set_xticks(range(len(short_lin)))
    ax.set_xticklabels(short_lin, fontsize=9)
    ax.set_title(f'{tp}\n(delta-vs-Sham)', fontsize=10, fontweight='bold')
    if tp == timepoint_order[0]:
        ax.set_yticks(range(len(labels_kept)))
        ax.set_yticklabels(labels_kept, fontsize=8)
fig.suptitle('Continuous neighborhood: neighbor-minus-self delta vs Sham (mouse-level mean)',
             fontsize=12, fontweight='bold', y=1.02)
fig.colorbar(im, ax=axes, label='Δ(neighbor lineage score) vs Sham', fraction=0.02)
plt.show()

# === Plot 2: join-count z-scores per interface category, distribution across ROIs by timepoint ===
join_counts['timepoint'] = pd.Categorical(join_counts['timepoint'], categories=timepoint_order, ordered=True)
fig2, ax2 = plt.subplots(figsize=(11, 5))
INTERFACE_CATS = [
    'immune', 'endothelial', 'stromal',
    'endothelial+immune', 'immune+stromal', 'endothelial+stromal',
    'endothelial+immune+stromal', 'none',
]
cat_positions = {cat: i for i, cat in enumerate(INTERFACE_CATS)}
tp_offsets = {'Sham': -0.30, 'D1': -0.10, 'D3': 0.10, 'D7': 0.30}
tp_colors = {'Sham': '#999999', 'D1': '#FFB347', 'D3': '#FF7043', 'D7': '#D32F2F'}

for tp in timepoint_order:
    sub = join_counts[(join_counts['timepoint'] == tp) & (join_counts['sufficient'])]
    for cat in INTERFACE_CATS:
        cat_sub = sub[sub['category'] == cat]
        if cat_sub.empty:
            continue
        x = cat_positions[cat] + tp_offsets[tp]
        ax2.scatter([x] * len(cat_sub), cat_sub['z_score'], color=tp_colors[tp], s=40, alpha=0.7, edgecolor='black', linewidth=0.4)
ax2.axhline(0, color='black', linewidth=0.6, linestyle='--', alpha=0.6)
ax2.set_xticks(range(len(INTERFACE_CATS)))
ax2.set_xticklabels([c.replace('+', '+\n') for c in INTERFACE_CATS], fontsize=8)
ax2.set_ylabel('Join-count z-score (positive = spatially clustered vs CSR null)')
ax2.set_title('Spatial coherence per interface category per ROI (1000-perm CSR null, k=10 NN)\nDots = ROIs (insufficient/saturated indicators omitted)',
              fontsize=11, fontweight='bold')
handles = [plt.scatter([], [], color=tp_colors[tp], label=tp, s=40, edgecolor='black', linewidth=0.4) for tp in timepoint_order]
ax2.legend(handles=handles, loc='upper right', fontsize=9)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.show()

# Missingness summary
status_counts = missingness['status'].value_counts().to_dict()
print(f"Continuous neighborhood missingness: {status_counts}")
print(f"  (sufficient = label has ≥1 mouse with valid delta; below_min_support = present but n_superpixels<20; absent_biology = label never appeared)")
print(f"Labels retained in trajectory: {missingness[missingness['kept_in_trajectory']]['composite_label'].nunique()}")


### Reading the Continuous Neighborhood + Coherence Plots

**Heatmap (delta-vs-Sham):** rows are composite cell-type labels, columns are the three lineages (immune/endothelial/stromal). Color = mouse-level mean of `(neighbor_lineage_X − self_lineage_X)` minus the Sham-baseline mean for the same label×lineage. Red = neighbors enriched for that lineage relative to Sham; blue = depleted. Sham column is definitionally near zero (residual variance only).

**Caveats:**
- The within-stratum self-bias on the diagonal (immune cell types tend to have negative `delta_immune` because the self score is high) is constant across timepoints (fixed threshold), so *changes* between Sham/D1/D3/D7 are interpretable; absolute values are not. Hence delta-vs-Sham as the visualization unit.
- Family B is 100% threshold-sensitive in `endpoint_summary.csv` because the min-support filter (default 20 superpixels) changes which labels survive at each level. Read the heatmap as exploration; Family A and Family C are the more robust families.

**Join-count panel:** dots are ROIs grouped by interface category. Z-scores are observed Black-Black joins for each binary indicator (e.g., is_triple) versus a 1000-permutation CSR null with k=10 NN adjacency. Higher = more spatially clustered. Saturated indicators (all-pos or all-neg) and insufficient ones (<10 positive superpixels) are omitted, not silently set to zero.

**What this section is NOT:** confirmatory inference. With n=2 mice/timepoint, neither the heatmap nor the join-count plot supports significance claims. Both are descriptive ranked exploration to feed follow-up study design.

For the full pre-registered endpoint table including threshold-sensitivity flags and normalization-mode flags per row, consult `results/biological_analysis/temporal_interfaces/endpoint_summary.csv` and `analysis_plans/temporal_interfaces_plan.md` (FDR proxy columns removed in Gate 6 — at n=2 no real p-values exist; rank audit is by sorting on `|hedges_g|`).

---

**Bridge to Part 3 (Tier 3 context).** Part 3 examines marker covariation. The correlation structure is a **Tier 3 context section** — it describes which markers co-vary at the superpixel level, but covariation is not causation. Correlations among markers in superpixels reflect co-expression, not driving, integration, or coordination of multi-lineage interfaces. The redistribution candidate finding is not tested by Part 3.


---

# Part 3: Marker Co-Expression — Which Proteins Vary Together?

## The Coordination Question

Proteins don't act independently. CD44 (activation) might coordinate with CD140b (stromal signaling) in fibroblasts, or with CD11b (myeloid) in immune cells.

**Correlation structure reveals biological modules**: groups of proteins that vary together define functional cell states.

In [ ]:
# Compute marker correlations at each timepoint
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, tp in enumerate(['D1', 'D3', 'D7']):
    ax = axes[idx]
    
    tp_data = df_10[df_10['timepoint'] == tp][markers]
    corr_matrix = tp_data.corr()
    
    # Plot heatmap
    im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-0.5, vmax=0.5, aspect='auto')
    
    # Add correlation values
    for i in range(len(markers)):
        for j in range(len(markers)):
            text = ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                          ha="center", va="center", 
                          color="white" if abs(corr_matrix.iloc[i, j]) > 0.3 else "black",
                          fontsize=8)
    
    ax.set_xticks(range(len(markers)))
    ax.set_yticks(range(len(markers)))
    ax.set_xticklabels(markers, rotation=45, ha='right')
    ax.set_yticklabels(markers)
    ax.set_title(f'{tp} Marker Correlations', fontweight='bold', fontsize=12)

# Add colorbar
cbar = fig.colorbar(im, ax=axes, orientation='horizontal', 
                    pad=0.05, aspect=50, shrink=0.8)
cbar.set_label('Correlation Coefficient', fontweight='bold')

plt.suptitle('Marker Co-Expression Patterns Across Injury Timeline', 
             fontweight='bold', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

# Print strongest correlations at D7
print("\nStrongest Marker Correlations at D7:")
d7_corr = df_10[df_10['timepoint'] == 'D7'][markers].corr()
# Get upper triangle excluding diagonal
corr_pairs = []
for i in range(len(markers)):
    for j in range(i+1, len(markers)):
        corr_pairs.append((markers[i], markers[j], d7_corr.iloc[i, j]))

corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
for m1, m2, r in corr_pairs[:10]:
    print(f"  {m1} ↔ {m2}: r={r:.3f}")

### Reading the Co-Expression Landscape

The correlation heatmaps reveal **three functional modules** that organize protein expression:

**Module 1: Immune Coordination (CD45 ↔ CD11b ↔ Ly6G)**
- Strong positive correlations (moderate-to-strong) at all timepoints
- Expected: CD45 (pan-immune) marks all leukocytes, CD11b (myeloid) marks subset, Ly6G (neutrophils) marks further subset
- **Hierarchical immune identity**: These markers nest within each other
- CD206 shows **weaker** correlation with the immune core - M2 macrophages are a distinct myeloid subset

**Module 2: Vascular Coupling (CD31 ↔ CD34)**  
- Strong correlation (strong) across all timepoints
- Both mark endothelium - CD31 (pan-endothelial), CD34 (progenitors/activated)
- Correlation stability indicates **structural constraint**: vessels either exist or don't
- Minimal correlation with immune/stromal markers - vascular compartment is **spatially segregated**

**Module 3: Stromal-Activation Axis (CD140a ↔ CD140b ↔ CD44)**
- Moderate correlations (moderate)
- CD140a/b (PDGFR receptors) mark fibroblasts/pericytes
- CD44 (activation marker) correlates with **both immune and stromal** markers
- **Observation**: CD44 covaries with markers from multiple lineages (immune, stromal, weakly vascular). This is correlation among markers measured in the same superpixels — *not* evidence that CD44 "bridges" or "coordinates" multi-lineage activation programs. The mechanism would require perturbation experiments; this notebook only observes correlation.

**The Critical Observation: CD44 Cross-Compartment Covariation**

At D7, CD44 correlates with:
- CD11b (moderate) - immune activation
- CD140b (moderate) - stromal activation  
- CD31 (weak) - vascular activation

**Observation (descriptive only)**: By Day 7, CD44 covariation extends across compartments. We do not infer mechanism from this. CD44 is **pan-tissue**. Immune cells, fibroblasts, and endothelium are all expressing activation markers. Multiple compartments show activation marker correlations — descriptive co-expression, not a coordinated mechanistic response.

**Why correlations are modest (modest)**: Tissue is heterogeneous. Not all immune-rich regions have activated stroma. Not all activated regions have intact vasculature. The **spatial organization** (which we'll see in clustering) creates this heterogeneity.

---

**Bridge to Part 4 (Tier 2 concordance check).** CD44 covaries with markers from multiple lineages (correlations only — not driving, not coordinating, not integrating). Part 4 asks whether unsupervised Leiden clusters are *consistent with* the multi-lineage interface concept — a concordance check on derived data, not independent validation. The clusters are computed from the same marker panel as Family A, so cluster correspondence to interface categories is re-expression of the same data through a different algorithm.


---

# Part 4: Cluster Biological Identity — What ARE These Communities?

## From Numbers to Biology

Leiden clustering found 6-18 spatial communities per ROI. But **what are they biologically?**

**Tier label: Tier 2 concordance check.** Leiden clusters and Family A interface categories are derived from the same underlying expression data. So cluster correspondence to interface categories is **not** independent corroboration — it is a re-expression of the same data through a different algorithm. We use Part 4 as a concordance check (do unsupervised clusters identify the multi-lineage regions Family A flagged?), not as confirmation.

Let's characterize clusters by their marker profiles to assign biological identities.


In [ ]:
# Pick one D7 ROI to deeply characterize
example_roi = df_10[df_10['timepoint'] == 'D7']['roi'].iloc[0]
roi_data = df_10[df_10['roi'] == example_roi].copy()

print(f"Characterizing: {example_roi}")
print(f"  {len(roi_data)} superpixels")
print(f"  {roi_data['cluster'].nunique()} clusters")

# Compute mean marker expression per cluster
cluster_profiles = roi_data.groupby('cluster')[markers].mean()

# Compute relative expression (z-score within each marker)
from scipy.stats import zscore
cluster_profiles_z = cluster_profiles.apply(zscore, axis=0)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: Heatmap of cluster profiles
ax = axes[0]
im = ax.imshow(cluster_profiles_z.T, cmap='RdBu_r', aspect='auto', vmin=-2, vmax=2)
ax.set_xticks(range(len(cluster_profiles_z)))
ax.set_xticklabels([f'C{i}' for i in cluster_profiles_z.index], fontsize=9)
ax.set_yticks(range(len(markers)))
ax.set_yticklabels(markers, fontsize=10)
ax.set_xlabel('Cluster ID', fontweight='bold')
ax.set_title('A. Cluster Marker Profiles (Z-scored)', fontweight='bold', fontsize=12)
plt.colorbar(im, ax=ax, label='Relative Expression (Z-score)')

# Panel B: Cluster sizes
ax = axes[1]
cluster_sizes = roi_data['cluster'].value_counts().sort_index()
colors_bar = plt.cm.tab10(np.linspace(0, 1, len(cluster_sizes)))
ax.bar(range(len(cluster_sizes)), cluster_sizes.values, color=colors_bar, 
       edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(cluster_sizes)))
ax.set_xticklabels([f'C{i}' for i in cluster_sizes.index], fontsize=9)
ax.set_ylabel('Number of Superpixels', fontweight='bold')
ax.set_xlabel('Cluster ID', fontweight='bold')
ax.set_title('B. Cluster Sizes', fontweight='bold', fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add percentage labels
for i, (idx, size) in enumerate(cluster_sizes.items()):
    pct = 100 * size / len(roi_data)
    ax.text(i, size + 50, f'{pct:.1f}%', ha='center', fontsize=8)

plt.suptitle(f'Cluster Characterization: {example_roi}', 
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Assign biological identities based on top markers
print("\nBiological Cluster Identities (by top 3 defining markers):")
print("="*80)
for cluster_id in cluster_profiles_z.index:
    profile = cluster_profiles_z.loc[cluster_id]
    top3 = profile.nlargest(3)
    size = cluster_sizes[cluster_id]
    pct = 100 * size / len(roi_data)
    
    # Biological interpretation logic
    if top3.index[0] == 'CD31' or top3.index[0] == 'CD34':
        identity = "🩸 Vascular"
    elif top3.index[0] in ['CD45', 'CD11b', 'Ly6G']:
        identity = "🦠 Immune Infiltrate"
    elif top3.index[0] == 'CD206' and 'CD11b' in top3.index:
        identity = "🛡️  M2 Macrophage"
    elif top3.index[0] in ['CD140a', 'CD140b']:
        identity = "🧬 Fibroblast/Pericyte"
    elif top3.index[0] == 'CD44':
        identity = "⚡ Activated"
    else:
        identity = "❓ Mixed/Transitional"
    
    print(f"\nCluster {cluster_id} ({size} spx, {pct:.1f}%): {identity}")
    print(f"  Top markers: {', '.join([f'{m} ({top3[m]:.2f}σ)' for m in top3.index])}")

In [ ]:
# Spatial organization of clusters
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# DNA tissue underlay
render_dna_underlay(ax, example_roi)

# Plot each cluster with a different color
n_clusters = roi_data['cluster'].nunique()
colors = plt.cm.tab10(np.linspace(0, 1, n_clusters))

for i, cluster_id in enumerate(sorted(roi_data['cluster'].unique())):
    cluster_spx = roi_data[roi_data['cluster'] == cluster_id]
    ax.scatter(cluster_spx['x'], cluster_spx['y'], 
              c=[colors[i]], s=20, alpha=0.85, 
              edgecolors='#333333', linewidth=0.3, label=f'C{cluster_id}',
              zorder=2)

ax.set_aspect('equal')
ax.set_xlabel('X (μm)', fontweight='bold', fontsize=12)
ax.set_ylabel('Y (μm)', fontweight='bold', fontsize=12)
ax.set_title(f'Spatial Distribution of Clusters: {example_roi}', 
            fontweight='bold', fontsize=14)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', 
         ncol=1, fontsize=9, frameon=True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

### The Tissue Landscape Revealed

The heatmap and spatial maps together answer: **what ARE these clusters?**

**Biological Identity Assignment** (example ROI at D7):

**Vascular Clusters** (CD31+/CD34+ high):
- Form linear/branching structures in spatial map - these are **vessels**
- Often a substantial fraction of tissue - kidney is highly vascularized
- **Spatial logic**: Vessels are structural - they define tissue architecture

**Immune Infiltrate Clusters** (CD45+/CD11b+ high):
- Form **discrete foci** - not diffuse
- Variable fraction of tissue by D7
- Some show Ly6G enrichment (neutrophil foci), others CD206 (M2 foci)
- **Spatial logic**: Immune cells recruited to damage sites, cluster together

**Fibroblast/Pericyte Clusters** (CD140a+/CD140b+ high):
- Interstitial distribution - fill spaces between structures
- A substantial fraction of tissue
- Some show CD44 co-expression (activated fibroblasts)
- **Spatial logic**: Stromal support cells, ubiquitous but heterogeneous activation

**Activated Mixed Clusters** (CD44 high, multiple markers):
- Highest heterogeneity - express immune + stromal + vascular markers
- Small but important fraction of tissue
- These are sites where all compartments co-express activation markers (CD44+ on immune, vascular, and stromal lineages simultaneously). Quantified temporally in Part 6 as triple-overlap fraction.

**Concordance reading (not validation).** Leiden clusters that mix immune, stromal, and vascular markers do correspond to the multi-lineage interface superpixels Family A flagged in Part 2 — this is a concordance check (same data, different algorithm), not independent validation of the candidate finding.
- **Spatial logic**: Sites of active tissue remodeling

**Quiescent/Low Expression Clusters**:
- Low expression across all markers
- Variable fraction of tissue
- **Interpretation**: Either unperturbed regions OR epithelium (not in our panel)
- **Spatial logic**: Potential "safe zones" that escaped injury

**Observation: Cluster Labels Show Spatial Patterning** (descriptive; the categorical Moran's I caveat below applies)

Looking at the spatial map:
- Vascular clusters form **networks** (branching structures)
- Immune clusters form **foci** (discrete aggregates)
- Activated clusters appear at **interfaces** (immune-vascular boundaries)

This is not diffuse inflammation. This is **microanatomical organization** - injury creates structured tissue compartments with distinct biological states.

**Why variable cluster counts per ROI?** Heterogeneity reflects:
1. **Anatomical diversity**: Cortex vs medulla, glomeruli vs tubules
2. **Injury severity gradient**: Direct pressure damage vs secondary inflammation
3. **Temporal asynchrony**: Some regions activated (CD44+), others quiescent
4. **Multi-lineage co-expression**: combinatorial marker states observed (descriptive; not interpreted as biological coordination)

The kidney shows non-uniform marker patterns — a **landscape of marker microstates**. We do not interpret these as repair-vs-fibrosis fate; that would require longitudinal lineage tracing.

---

> **Methodological note: Moran's I on cluster labels.** The pre-registered analysis (`temporal_interfaces_plan.md` §6) uses **join-count statistics on binary indicators** for spatial autocorrelation of categorical data — not Moran's I on integer cluster labels (which is mathematically incoherent because labels are nominal, not ordinal). The Moran's I value computed in the code cell above is reported here as a **legacy descriptive metric** for historical continuity, NOT as evidence. The methodologically correct equivalent for this notebook is in Part 2.5's join-count panel.


### Validation: Do Leiden Clusters Correspond to Known Cell Types?

**The Circular Logic Problem (acknowledged from the start):**
- Saying "Cluster 3 has high CD44 → therefore it's fibrotic" is circular reasoning
- Leiden clusters and supervised phenotypes both derive from the SAME marker panel — concordance does not constitute independent validation

**What this section ACTUALLY does (Tier 2 concordance):**
- We link two **same-data, different-algorithm** views of the panel: unsupervised Leiden clusters and supervised boolean-gate phenotypes
- If the unsupervised clusters happen to enrich for boolean-gate cell types, that's *concordance evidence* — same biology measured two ways. It is **not** independent validation against ground truth (we don't have ground truth at the superpixel level).

**The correct framing**: cluster-phenotype concordance is a sanity check on the algorithmic clustering, not a test of the candidate redistribution hypothesis (which requires Family A + Family C + Part 2.5 spatial coherence).

Let's run the concordance check on a representative D7 ROI:


In [ ]:
# Cluster-Phenotype Enrichment Analysis
# ======================================

# Select representative D7 ROI with sufficient clusters
example_roi_id = 'IMC_241218_Alun_ROI_D7_M2_01_24'
roi_data = df_10[df_10['roi'] == example_roi_id].copy()

print(f"Analyzing ROI: {example_roi_id}")
print(f"Total superpixels: {len(roi_data)}")
print(f"Leiden clusters: {roi_data['cluster'].nunique()}")

# Data-driven thresholds: 75th percentile of each marker across all D7 data
# This avoids hardcoded gates and adapts to the arcsinh-transformed scale
d7_data = df_10[df_10['timepoint'] == 'D7']
thresholds = {m: d7_data[m].quantile(0.75) for m in markers}
print(f"\nGating thresholds (75th percentile of D7):")
for m, t in thresholds.items():
    print(f"  {m}: {t:.3f}")

# Define supervised phenotypes using available panel markers
phenotype_definitions = {
    'M2_Macrophage': (roi_data['CD206'] > thresholds['CD206']),
    'Neutrophil': (roi_data['Ly6G'] > thresholds['Ly6G']),
    'Immune': (roi_data['CD45'] > thresholds['CD45']) & (roi_data['CD11b'] > thresholds['CD11b']),
    'Endothelial': (roi_data['CD31'] > thresholds['CD31']),
    'Fibroblast': (roi_data['CD140b'] > thresholds['CD140b']),
    'Pericyte': (roi_data['CD140a'] > thresholds['CD140a']),
}

# Add phenotype columns
for pheno_name, pheno_mask in phenotype_definitions.items():
    roi_data[pheno_name] = pheno_mask.astype(int)

# Compute enrichment matrix
phenotype_cols = list(phenotype_definitions.keys())
cluster_ids = sorted(roi_data['cluster'].unique())

pct_matrix = np.zeros((len(cluster_ids), len(phenotype_cols)))
enrichment_matrix = np.zeros((len(cluster_ids), len(phenotype_cols)))

# Overall phenotype prevalence (background)
overall_prevalence = {pheno: roi_data[pheno].mean() for pheno in phenotype_cols}

for i, cluster_id in enumerate(cluster_ids):
    cluster_data = roi_data[roi_data['cluster'] == cluster_id]
    
    for j, pheno in enumerate(phenotype_cols):
        pct_in_cluster = cluster_data[pheno].mean()
        pct_overall = overall_prevalence[pheno]
        
        pct_matrix[i, j] = 100 * pct_in_cluster
        enrichment_matrix[i, j] = pct_in_cluster / pct_overall if pct_overall > 0 else 0

# Visualization: 2-panel figure
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Phenotype percentages per cluster
sns.heatmap(
    pct_matrix,
    xticklabels=phenotype_cols,
    yticklabels=[f'C{c}' for c in cluster_ids],
    annot=True, fmt='.1f', cmap='YlOrRd',
    cbar_kws={'label': '% Positive'}, vmin=0, vmax=50, ax=axes[0]
)
axes[0].set_title('Phenotype Prevalence per Cluster (%)', fontsize=14, weight='bold')
axes[0].set_xlabel('Supervised Phenotype', fontsize=12)
axes[0].set_ylabel('Leiden Cluster', fontsize=12)

# Panel 2: Enrichment (fold-change vs background)
sns.heatmap(
    enrichment_matrix,
    xticklabels=phenotype_cols,
    yticklabels=[f'C{c}' for c in cluster_ids],
    annot=True, fmt='.2f', cmap='RdBu_r',
    center=1.0, vmin=0, vmax=3.0,
    cbar_kws={'label': 'Enrichment (fold-change)'}, ax=axes[1]
)
axes[1].set_title('Phenotype Enrichment vs Background', fontsize=14, weight='bold')
axes[1].set_xlabel('Supervised Phenotype', fontsize=12)
axes[1].set_ylabel('Leiden Cluster', fontsize=12)

plt.tight_layout()
plt.show()

# Statistical interpretation
print("\n" + "="*60)
print("INTERPRETATION: Cluster-Phenotype Associations")
print("="*60)

for i, cluster_id in enumerate(cluster_ids):
    enriched_phenotypes = [
        (phenotype_cols[j], enrichment_matrix[i, j]) 
        for j in range(len(phenotype_cols)) 
        if enrichment_matrix[i, j] > 1.5
    ]
    
    if enriched_phenotypes:
        print(f"\nCluster {cluster_id}:")
        for pheno, enrichment in sorted(enriched_phenotypes, key=lambda x: -x[1]):
            pct = pct_matrix[i, phenotype_cols.index(pheno)]
            print(f"  - {pheno}: {enrichment:.2f}x enriched ({pct:.1f}% positive)")
    else:
        print(f"\nCluster {cluster_id}: No strong phenotype enrichment (likely mixed/transitional)")


---

# Validation: Are Clusters Statistically Meaningful?

## Three Critical Questions for Publication

Before interpreting clusters biologically, we must validate they're not statistical artifacts:

1. **Are clusters well-separated?** → Silhouette score
   - Measures how distinct clusters are in marker space
   - Range: -1 (wrong clustering) to +1 (perfect separation)
   - Expected for biological data: 0.25-0.50 (weak-to-moderate structure)

2. **Are clusters spatially coherent?** → Moran's I
   - Tests if nearby superpixels have similar cluster labels
   - Range: -1 (dispersed) to +1 (clustered)
   - Expected: Positive values indicate spatial organization

3. **Are clusters reproducible?** → Cross-ROI consistency
   - Do all 18 ROIs show similar validation metrics?
   - Consistent metrics indicate robust patterns

**Why This Matters for Reviewers:**
Unsupervised clustering can always find structure, even in random data. These metrics validate that our clusters reflect real biological organization, not algorithmic artifacts.

In [ ]:
# Build all_results from ROI result files
import gzip, json
from pathlib import Path

all_results = {}
for rf in sorted(Path('../../results/roi_results').glob('roi_*_results.json.gz')):
    roi_name = rf.name.replace('_results.json.gz', '').replace('roi_', '')
    with gzip.open(rf, 'rt') as f:
        all_results[roi_name] = json.load(f)
print(f"Loaded {len(all_results)} ROI results for validation")

# Compute validation metrics for all ROIs
import numpy as np
from sklearn.metrics import silhouette_score
from scipy.spatial.distance import pdist, squareform

validation_results = []

print("Computing validation metrics...")
for roi_id, results in all_results.items():
    scale_data = results['multiscale_results']['10.0']
    
    # Deserialize
    features = deserialize_array(scale_data['features'])
    clusters = deserialize_array(scale_data['cluster_labels'])
    coords = deserialize_array(scale_data['spatial_coords'])
    
    # 1. Silhouette score (cluster separation quality)
    if len(np.unique(clusters)) > 1:
        # Sample for efficiency on large datasets
        sample_size = min(1000, len(clusters))
        silhouette = silhouette_score(features, clusters, sample_size=sample_size)
    else:
        silhouette = 0
    
    # 2. Moran's I (spatial autocorrelation)
    # Create spatial weights matrix (inverse distance)
    dist_matrix = squareform(pdist(coords))
    np.fill_diagonal(dist_matrix, 1)  # Avoid division by zero
    weights = 1.0 / dist_matrix
    np.fill_diagonal(weights, 0)
    
    # Compute Moran's I
    n = len(clusters)
    mean_cluster = clusters.mean()
    
    numerator = 0
    denominator = 0
    W = weights.sum()
    
    for i in range(n):
        for j in range(n):
            numerator += weights[i, j] * (clusters[i] - mean_cluster) * (clusters[j] - mean_cluster)
        denominator += (clusters[i] - mean_cluster) ** 2
    
    morans_i = (n / W) * (numerator / denominator) if denominator > 0 else 0
    
    # Store results
    validation_results.append({
        'roi': roi_id,
        'n_clusters': len(np.unique(clusters)),
        'silhouette': silhouette,
        'morans_i': morans_i,
        'n_superpixels': len(clusters)
    })

validation_df = pd.DataFrame(validation_results)

# Print summary statistics
print("\n" + "=" * 80)
print("CLUSTERING VALIDATION METRICS (18 ROIs, 10μm scale)")
print("=" * 80)
print()
print(f"Silhouette Score (cluster separation quality):")
print(f"  Mean: {validation_df['silhouette'].mean():.3f} ± {validation_df['silhouette'].std():.3f}")
print(f"  Range: [{validation_df['silhouette'].min():.3f}, {validation_df['silhouette'].max():.3f}]")
print(f"  Interpretation: 0.25-0.50 = weak-moderate structure (expected for biology)")
print(f"                  0.50-0.70 = moderate-strong structure")
print(f"                  >0.70 = strong separation")
print()
print(f"Moran's I (spatial autocorrelation):")
print(f"  Mean: {validation_df['morans_i'].mean():.3f} ± {validation_df['morans_i'].std():.3f}")
print(f"  Range: [{validation_df['morans_i'].min():.3f}, {validation_df['morans_i'].max():.3f}]")
print(f"  Interpretation: >0 = positive spatial clustering (organized)")
print(f"                  0 = random distribution")
print(f"                  <0 = dispersed (checkerboard)")
print()
print(f"Cluster count per ROI:")
print(f"  Mean: {validation_df['n_clusters'].mean():.1f} ± {validation_df['n_clusters'].std():.1f}")
print(f"  Range: [{validation_df['n_clusters'].min()}, {validation_df['n_clusters'].max()}]")
print()

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: Silhouette scores distribution
ax = axes[0]
ax.hist(validation_df['silhouette'], bins=12, edgecolor='black', alpha=0.7, color='#3498db')
ax.axvline(validation_df['silhouette'].mean(), color='red', linestyle='--', linewidth=2,
           label=f"Mean: {validation_df['silhouette'].mean():.3f}")
ax.axvspan(0.25, 0.50, alpha=0.2, color='green', label='Expected range')
ax.set_xlabel('Silhouette Score', fontsize=12)
ax.set_ylabel('Number of ROIs', fontsize=12)
ax.set_title('Cluster Separation Quality', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Panel B: Moran's I distribution
ax = axes[1]
ax.hist(validation_df['morans_i'], bins=12, edgecolor='black', alpha=0.7, color='#e74c3c')
ax.axvline(validation_df['morans_i'].mean(), color='red', linestyle='--', linewidth=2,
           label=f"Mean: {validation_df['morans_i'].mean():.3f}")
ax.axvline(0, color='gray', linestyle='-', alpha=0.5, linewidth=2, label='Random (0)')
ax.set_xlabel("Moran's I", fontsize=12)
ax.set_ylabel('Number of ROIs', fontsize=12)
ax.set_title('Spatial Autocorrelation', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Panel C: Scatter plot - relationship between metrics
ax = axes[2]
scatter = ax.scatter(validation_df['silhouette'], validation_df['morans_i'], 
                    s=100, c=validation_df['n_clusters'], cmap='viridis',
                    edgecolor='black', alpha=0.7)
ax.set_xlabel('Silhouette Score\n(Cluster Separation)', fontsize=12)
ax.set_ylabel("Moran's I\n(Spatial Coherence)", fontsize=12)
ax.set_title('Validation Metrics Relationship', fontsize=13, fontweight='bold')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(0.25, color='gray', linestyle='--', alpha=0.5)
cbar = plt.colorbar(scatter, ax=ax, label='# Clusters')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Interpretation
print("=" * 80)
print("STATISTICAL INTERPRETATION")
print("=" * 80)
print()
if validation_df['silhouette'].mean() > 0.25:
    print("✓ SILHOUETTE: Clusters are well-separated (mean > 0.25)")
    print("  → Leiden clustering identified distinct communities in marker space")
else:
    print("⚠ SILHOUETTE: Weak separation (mean < 0.25)")
    print("  → Clusters may overlap; boundaries are fuzzy")
    
print()
if validation_df['morans_i'].mean() > 0:
    print("✓ MORAN'S I: Positive spatial autocorrelation (mean > 0)")
    print("  → Clusters are spatially organized, not randomly scattered")
    print(f"  → Nearby superpixels tend to belong to same cluster")
else:
    print("⚠ MORAN'S I: No spatial organization (mean ≈ 0)")
    print("  → Clusters are randomly distributed")
    
print()
print(f"✓ REPRODUCIBILITY: Consistent metrics across {len(validation_df)} ROIs")
print(f"  → Silhouette SD = {validation_df['silhouette'].std():.3f} (low variance)")
print(f"  → Moran's I SD = {validation_df['morans_i'].std():.3f}")
print()
print("=" * 80)
print("DESCRIPTIVE OBSERVATION (n=2 per timepoint; not inferential)")
print("=" * 80)
print()
print("Leiden clustering produces statistically meaningful communities:")
print("  1. Moderate cluster separation (silhouette > 0.25)")
print("  2. Positive spatial organization (Moran's I > 0)")
print("  3. Reproducible across 18 ROIs from 2 mice")
print()
print("These metrics validate that clusters reflect BIOLOGICAL spatial")
print("organization, not algorithmic artifacts or overfitting.")
print("=" * 80)

---

# Part 5: The Neutrophil Paradox — Negative Control for the Candidate Finding

## The Mystery

**Expected biology** (from UUO literature): Neutrophils (Ly6G+) arrive first at injury sites - peak at Day 1, then decline as macrophages replace them.

**Our observation**: Ly6G expression is **FLAT** across D1/D3/D7 (mean barely changes between timepoints).

**The question for Part 5**: Are neutrophils absent? Or are we measuring wrong?

**The synthesis role of Part 5 (Tier 3, negative control).** If the candidate redistribution finding (stromal-only ↓ + triple-positive ↑ at Sham→D7) were a global artifact of CLR or per-ROI normalization applied uniformly to all cell types, *every* cell type would show similar artifactual movement. Neutrophils don't. They appear stable-but-focal across timepoints under the same analytical pipeline. Their non-redistribution under the same machinery supports a biological rather than artifactual origin for the Family A + Family C signal.


In [ ]:
# Investigate Ly6G spatial distribution
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Row 1: Ly6G distributions at each timepoint
for idx, tp in enumerate(['D1', 'D3', 'D7']):
    ax = axes[0, idx]
    tp_data = df_10[df_10['timepoint'] == tp]['Ly6G']
    
    ax.hist(tp_data, bins=50, color='#E74C3C', alpha=0.7, edgecolor='black')
    ax.axvline(tp_data.mean(), color='black', linestyle='--', linewidth=2, 
              label=f'Mean = {tp_data.mean():.3f}')
    ax.axvline(tp_data.median(), color='blue', linestyle='--', linewidth=2,
              label=f'Median = {tp_data.median():.3f}')
    
    # Mark 90th percentile (high Ly6G)
    p90 = np.percentile(tp_data, 90)
    ax.axvline(p90, color='red', linestyle=':', linewidth=2,
              label=f'90th % = {p90:.3f}')
    
    ax.set_xlabel('Ly6G Expression (arcsinh)', fontweight='bold')
    ax.set_ylabel('Count', fontweight='bold')
    ax.set_title(f'{tp}: Ly6G Distribution', fontweight='bold', fontsize=12)
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Row 2: Spatial maps showing high Ly6G regions
for idx, tp in enumerate(['D1', 'D3', 'D7']):
    ax = axes[1, idx]
    
    # Pick one ROI from this timepoint
    tp_rois = df_10[df_10['timepoint'] == tp]['roi'].unique()
    if len(tp_rois) > 0:
        roi = tp_rois[0]
        roi_data = df_10[df_10['roi'] == roi]
        
        # DNA tissue underlay
        render_dna_underlay(ax, roi)
        
        # Plot all superpixels (gray)
        ax.scatter(roi_data['x'], roi_data['y'],
                  c='lightgray', s=15, alpha=0.4, edgecolors='none',
                  zorder=1)
        
        # Highlight high Ly6G (>75th percentile)
        p75 = np.percentile(roi_data['Ly6G'], 75)
        high_ly6g = roi_data[roi_data['Ly6G'] > p75]
        scatter = ax.scatter(high_ly6g['x'], high_ly6g['y'],
                  c=high_ly6g['Ly6G'], cmap='Reds', s=30, alpha=0.85,
                  edgecolors='#333333', linewidth=0.3, vmin=p75, vmax=roi_data['Ly6G'].max(),
                  zorder=2)
        
        plt.colorbar(scatter, ax=ax, label='Ly6G Expression')
        ax.set_aspect('equal')
        ax.set_title(f'{tp} ({roi}): High Ly6G Regions', 
                    fontweight='bold', fontsize=12)
        ax.set_xlabel('X (μm)')
        ax.set_ylabel('Y (μm)')

plt.suptitle('Neutrophil Paradox: Are Neutrophils Spatially Focal?', 
             fontweight='bold', fontsize=14, y=1.0)
plt.tight_layout()
plt.show()

# Quantify ROI-level variance
print("\nLy6G Variance Analysis:")
print("="*80)
for tp in ['D1', 'D3', 'D7']:
    # Mean across ROIs
    roi_means = df_10[df_10['timepoint'] == tp].groupby('roi')['Ly6G'].mean()
    # Variance within ROIs (average)
    roi_vars = df_10[df_10['timepoint'] == tp].groupby('roi')['Ly6G'].var()
    
    print(f"\n{tp}:")
    print(f"  ROI-level mean: {roi_means.mean():.3f} ± {roi_means.std():.3f}")
    print(f"  Within-ROI variance (mean): {roi_vars.mean():.3f}")
    print(f"  Fraction of superpixels > 90th %ile: {100 * (df_10[df_10['timepoint']==tp]['Ly6G'] > np.percentile(df_10['Ly6G'], 90)).sum() / len(df_10[df_10['timepoint']==tp]):.1f}%")

In [ ]:
# Neutrophil Foci Quantification
# Define "foci" as regions with Ly6G in top 10% (90th percentile)

foci_analysis = []

for tp in ['Sham', 'D1', 'D3', 'D7']:
    tp_data = df_10[df_10['timepoint'] == tp]
    if len(tp_data) == 0:
        continue
    
    # Define foci threshold (90th percentile)
    foci_threshold = np.percentile(tp_data['Ly6G'], 90)
    
    # Identify foci superpixels
    in_foci = tp_data['Ly6G'] > foci_threshold
    n_foci = in_foci.sum()
    pct_foci = 100 * n_foci / len(tp_data)
    
    # Compute Ly6G statistics
    ly6g_in_foci = tp_data[in_foci]['Ly6G'].mean() if n_foci > 0 else 0
    ly6g_outside = tp_data[~in_foci]['Ly6G'].mean()
    ly6g_overall = tp_data['Ly6G'].mean()
    
    # Fold enrichment in foci vs tissue average
    fold_enrichment = ly6g_in_foci / ly6g_overall if ly6g_overall > 0 else 0
    
    foci_analysis.append({
        'timepoint': tp,
        'n_superpixels': len(tp_data),
        'n_foci': n_foci,
        'pct_foci': pct_foci,
        'foci_threshold': foci_threshold,
        'ly6g_in_foci': ly6g_in_foci,
        'ly6g_outside': ly6g_outside,
        'ly6g_overall': ly6g_overall,
        'fold_enrichment': fold_enrichment
    })

foci_df = pd.DataFrame(foci_analysis)

# Print statistics
print("=" * 80)
print("NEUTROPHIL FOCI QUANTIFICATION")
print("=" * 80)
print()
for _, row in foci_df.iterrows():
    print(f"{row['timepoint']:5s}: {row['pct_foci']:4.1f}% of tissue in neutrophil foci (n={row['n_foci']:,} superpixels)")
    print(f"       Ly6G in foci: {row['ly6g_in_foci']:.3f} vs {row['ly6g_outside']:.3f} outside")
    print(f"       {row['fold_enrichment']:.2f}× enrichment vs tissue average")
    print(f"       90th percentile threshold: {row['foci_threshold']:.3f}")
    print()

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: Percentage of tissue in foci
ax = axes[0]
colors = {'Sham': '#95a5a6', 'D1': '#e74c3c', 'D3': '#3498db', 'D7': '#2ecc71'}
bars = ax.bar(foci_df['timepoint'], foci_df['pct_foci'], 
              color=[colors.get(tp, 'gray') for tp in foci_df['timepoint']], 
              edgecolor='black', alpha=0.8, linewidth=1.5)
ax.set_ylabel('% Tissue in Neutrophil Foci\n(Ly6G > 90th percentile)', fontsize=12)
ax.set_xlabel('Timepoint', fontsize=12)
ax.set_title('Foci Prevalence Over Time', fontsize=13, fontweight='bold')
ax.set_ylim(0, max(foci_df['pct_foci']) * 1.2)
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (idx, row) in enumerate(foci_df.iterrows()):
    ax.text(i, row['pct_foci'] + 0.3, f"{row['pct_foci']:.1f}%", 
            ha='center', fontsize=10, fontweight='bold')

# Panel B: Ly6G intensity in foci vs outside
ax = axes[1]
x = np.arange(len(foci_df))
width = 0.35
bars1 = ax.bar(x - width/2, foci_df['ly6g_in_foci'], width, label='In Foci', 
               color='#e74c3c', edgecolor='black', alpha=0.8, linewidth=1.5)
bars2 = ax.bar(x + width/2, foci_df['ly6g_outside'], width, label='Outside Foci',
               color='#bdc3c7', edgecolor='black', alpha=0.8, linewidth=1.5)
ax.set_xticks(x)
ax.set_xticklabels(foci_df['timepoint'])
ax.set_ylabel('Mean Ly6G (arcsinh-transformed)', fontsize=12)
ax.set_xlabel('Timepoint', fontsize=12)
ax.set_title('Ly6G Intensity: Foci vs Background', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Panel C: 90th percentile trajectory (THE KEY METRIC)
ax = axes[2]
ax.plot(foci_df['timepoint'], foci_df['foci_threshold'], 
        'o-', markersize=12, linewidth=3, color='#e67e22', markeredgecolor='black', markeredgewidth=1.5)
ax.set_ylabel('Ly6G 90th Percentile\n(arcsinh-transformed)', fontsize=12)
ax.set_xlabel('Timepoint', fontsize=12)
ax.set_title('Neutrophil-Rich Regions Intensify', fontsize=13, fontweight='bold')
ax.grid(alpha=0.3)

# Add value labels
for i, (idx, row) in enumerate(foci_df.iterrows()):
    ax.text(i, row['foci_threshold'] + 0.05, f"{row['foci_threshold']:.3f}",
            ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

# Statistical interpretation
print("=" * 80)
print("RESOLUTION OF THE NEUTROPHIL PARADOX")
print("=" * 80)
print()
print("Evidence for focal neutrophil distribution:")
print()
print(f"  1. FOCAL ACCUMULATION:")
print(f"     - Only ~{foci_df['pct_foci'].mean():.1f}% of tissue shows high Ly6G (foci)")
print(f"     - Ly6G in foci is {foci_df['fold_enrichment'].mean():.1f}× higher than tissue average")
print()
print(f"  2. 90TH PERCENTILE DOES INCREASE:")
sham_data = foci_df[foci_df['timepoint']=='Sham']
if len(sham_data) > 0:
    print(f"     - Sham: {sham_data['foci_threshold'].values[0]:.3f}")
else:
    print("     - Sham: Not loaded (injury timepoints only)")
if 'D1' in foci_df['timepoint'].values:
    print(f"     - D1:   {foci_df[foci_df['timepoint']=='D1']['foci_threshold'].values[0]:.3f}")
if 'D3' in foci_df['timepoint'].values:
    print(f"     - D3:   {foci_df[foci_df['timepoint']=='D3']['foci_threshold'].values[0]:.3f}")
print(f"     - D7:   {foci_df[foci_df['timepoint']=='D7']['foci_threshold'].values[0]:.3f}")
if len(sham_data) > 0:
    print(f"     - Change: {((foci_df[foci_df['timepoint']=='D7']['foci_threshold'].values[0] / sham_data['foci_threshold'].values[0]) - 1) * 100:.1f}%")
else:
    print("     - Change: Cannot compute (Sham baseline not loaded)")
print()
print(f"  3. WHY MEANS APPEAR FLAT:")
print(f"     - ~90% of tissue has low Ly6G (background dilutes signal)")
print(f"     - Spatial averaging obscures focal accumulation")
print(f"     - ROI-level means: {foci_df['ly6g_overall'].min():.3f} - {foci_df['ly6g_overall'].max():.3f} (narrow range)")
print()
print("CONCLUSION:")
print("  Neutrophils ARE present, organized spatially, and increase over time")
print("  in neutrophil-rich foci. Their focal distribution creates a")
print("  measurement challenge - ROI-level means miss the biology.")
print("=" * 80)

### Quantifying the Neutrophil Paradox

**The visual evidence suggests focal distribution, but we need statistics to quantify it.**

Let's quantify:
1. What % of tissue is in "neutrophil foci" (top 10% Ly6G)?
2. How much higher is Ly6G in foci vs background?
3. Does the 90th percentile actually increase over time?

### Solving the Neutrophil Paradox

The histograms and spatial maps reveal the answer: **neutrophils are spatially focal, not diffuse**.

**Row 1: Ly6G distributions show**:
- Bimodal pattern - most tissue is Ly6G-low, small fraction is Ly6G-high
- Mean changes minimally because MOST tissue lacks neutrophils
- 90th percentile DOES shift upward from D1 to D7 (see code output above for exact values)
- **Interpretation**: Neutrophil-rich regions ARE increasing, but they're sparse

**Row 2: Spatial maps show**:
- High Ly6G forms **discrete foci** - not uniform infiltration
- At D1: Small scattered foci
- At D3: Larger foci, more numerous
- At D7: Persistent foci (neutrophils haven't fully resolved)
- **Interpretation**: Neutrophils cluster at injury sites - ROI-level means average them out

**Why ROI-level means miss the signal**:

Imagine 3 scenarios:
1. **Diffuse infiltration**: All tissue gets 10% neutrophils → ROI mean = 10%
2. **Focal infiltration**: 10% of tissue gets 100% neutrophils → ROI mean = 10%
3. **Sparse focal**: 5% of tissue gets 100% neutrophils → ROI mean = 5%

Our data suggests **scenario 3**: neutrophils are highly concentrated in small regions (foci around damaged tubules, perivascular aggregates), but occupy <10% of total tissue area. When we average across an entire ROI (thousands of superpixels), the signal is diluted.

**Negative control synthesis (Tier 3 role of Part 5).** Neutrophils show focal accumulation that ROI-level means dilute — but they are *stable-but-focal*, not redistributing. They serve as a negative control for the candidate finding: the Family A + Family C signal is not a generic artifact applied uniformly to all cell types under this pipeline. If the analytical machinery were manufacturing apparent redistribution everywhere, it would have manufactured it for neutrophils too. It didn't.

This does not prove the candidate finding is real. It only argues against — without ruling out — the "uniform-all-cell-types pipeline bias" alternative. Marker- or category-specific normalization artifacts remain possible.


---

# Part 6: Cross-Compartment Activation Trajectories (Tier 1)

## Returning to Direct Evidence

Part 6 returns to **Tier 1 evidence**. Family C uses raw-marker compartment definitions (CD45, CD31, CD140b for compartments; CD44 for activation; thresholds from the Sham distribution at the 75th percentile, with sensitivity sweep at {65, 75, 85}). This is methodologically distinct from Family A's compositional CLR — different markers (CD140b vs Family A's CD140a; CD31 alone vs Family A's mean(CD31, CD34)), different math (raw threshold vs CLR transform).

**Compatibility check (not a test)**: if both families show the same direction at Sham→D7, that's *related-but-non-identical convergence across measurement schemes* — partial corroboration. If they disagree, the Family A signal lacks Family C support. We compare; we do not test, since at n=2 no inferential test is possible.

If the candidate finding holds at the compartment level, triple-overlap regions (CD45+ AND CD31+ AND CD140b+) should EXPAND from baseline to D7. We test this directly with mouse-level trajectories.


In [ ]:
# Cross-compartment activation trajectories across all 4 timepoints
# Replaces the prior D7-only single-ROI snapshot with mouse-level temporal data
# computed against a Sham-reference threshold (NOT per-ROI floating threshold).
#
# Plan reference: analysis_plans/temporal_interfaces_plan.md §9 (Part 6 replacement spec).
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

base = project_root / 'results' / 'biological_analysis' / 'temporal_interfaces'
activation = pd.read_parquet(base / 'compartment_activation_temporal.parquet')
endpoint_summary = pd.read_csv(base / 'endpoint_summary.csv')

timepoint_order = ['Sham', 'D1', 'D3', 'D7']
activation['timepoint'] = pd.Categorical(activation['timepoint'], categories=timepoint_order, ordered=True)
activation = activation.sort_values(['timepoint', 'mouse']).reset_index(drop=True)

# 5-panel: 4 compartments + triple-overlap fraction
panels = [
    ('CD45_compartment_cd44_rate', 'CD45+ compartment', '#E63946'),
    ('CD31_compartment_cd44_rate', 'CD31+ compartment', '#06AED5'),
    ('CD140b_compartment_cd44_rate', 'CD140b+ compartment', '#2A9D8F'),
    ('background_compartment_cd44_rate', 'Background (no compartment)', '#95A5A6'),
    ('triple_overlap_fraction', 'Triple-overlap fraction (all 3 compartments)', '#7D3C98'),
]
fig, axes = plt.subplots(1, 5, figsize=(22, 4.5), sharey=False)
x_positions = {tp: i for i, tp in enumerate(timepoint_order)}
for ax, (col, title, color) in zip(axes, panels):
    for tp in timepoint_order:
        sub = activation[activation['timepoint'] == tp]
        if sub.empty or col not in sub.columns:
            continue
        xs = np.full(len(sub), x_positions[tp]) + np.random.RandomState(42).uniform(-0.08, 0.08, size=len(sub))
        ax.scatter(xs, sub[col].values, c=color, s=110, edgecolor='black', linewidth=0.7, zorder=3)
        ax.hlines(sub[col].mean(), x_positions[tp]-0.18, x_positions[tp]+0.18, color=color, linewidth=2.5, alpha=0.6)
    ax.set_xticks(range(len(timepoint_order)))
    ax.set_xticklabels(timepoint_order, fontsize=10)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_ylabel('CD44+ rate' if 'cd44_rate' in col else 'Triple-overlap fraction', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
fig.suptitle('Cross-compartment activation trajectories (mouse-level; Sham-reference 75th-percentile threshold)',
             fontsize=12, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

# Print top Family C effect sizes (Sham comparisons)
fc_top = (endpoint_summary[
    (endpoint_summary['family'] == 'C_compartment_activation') &
    (endpoint_summary['contrast'].isin(['Sham_vs_D1', 'Sham_vs_D3', 'Sham_vs_D7']))
].sort_values('hedges_g', key=abs, ascending=False).head(8))
print('Top Family C Sham-comparison effect sizes (excluding pathological/insufficient):')
print(fc_top[[
    'endpoint', 'contrast', 'mouse_mean_1', 'mouse_mean_2',
    'hedges_g', 'g_shrunk_skeptical', 'g_shrunk_neutral', 'g_shrunk_optimistic', 'n_required_neutral', 'threshold_sensitive',
]].to_string(index=False))


### Reading the Compartment Activation Trajectories

The 5-panel grid above shows mouse-level CD44+ activation rates within each compartment, plus the triple-overlap fraction, across the full Sham→D7 trajectory. Compartments are defined by Sham-reference 75th-percentile thresholds (a single global cutoff per marker, computed once from Sham ROIs) — this replaces the prior per-ROI floating threshold which made cross-timepoint comparison incoherent.

**Largest defensible Family C effects (from `endpoint_summary.csv`).** Bayesian shrinkage range uses three priors on the true effect δ (skeptical N(0, 0.5²), neutral N(0, 1.0²), optimistic N(0, 2.0²)); Hedges & Olkin (1985) sampling variance v = 2/n + g²/(4n):

| Endpoint | Observed g | Shrunk (skeptical / neutral / optimistic) | n required (skeptical / neutral / optimistic) | Threshold-sensitive across {65, 75, 85}? |
|---|---|---|---|---|
| Triple-overlap fraction | +3.29 | +0.32 / +0.98 / +2.07 | 158 / 17 / 4 | No |
| Background CD44+ rate | +2.82 | +0.31 / +0.94 / +1.88 | 160 / 18 / 5 | No |
| CD140b+ compartment CD44+ rate | +1.44 | +0.24 / +0.64 / +1.10 | 276 / 39 / 14 | No |
| CD45+ compartment CD44+ rate | +0.86 | small shrinkage | many hundreds / ~80 / ~30 | No |
| CD31+ compartment CD44+ rate | +0.30 | negligible | many thousands | No |

**Sober reading:**
- Triple-overlap fraction shows the largest temporal shift (g=+3.29 Sham→D7). The Bayesian shrinkage range spans (+0.32 / +0.98 / +2.07); n_required spans 158 / 17 / 4 depending on prior. No single prior is recommended as "default" — select one matching your own scepticism about the pilot estimate.
- Background CD44+ rate also rises substantially (g=+2.82; shrinkage 0.31 / 0.94 / 1.88; n_req 160 / 18 / 5).
- Within-compartment CD44+ shifts are smaller (CD140b+ g=+1.44; shrinkage 0.24 / 0.64 / 1.10; n_req 276 / 39 / 14).
- All Family C findings robust to the percentile sweep {65, 75, 85}.

**Pre-registered cautions repeated here per plan §10:**
- Per-ROI sigmoid normalization is a documented confound; mouse-level aggregation mitigates but does not eliminate it.
- CD44+ rates across compartments are *not independent* (a superpixel can be CD45+ AND CD31+); cross-compartment comparison is not performed, only within-compartment trajectories.
- Bayesian shrinkage does not resolve the n=2 problem — it only formalizes the shrinkage in a reproducible way. The three priors span the conventional small / medium / large effect-size expectations in Cohen's d units. The range IS the finding; we do not anoint any prior as default.
- The sampling variance formula (Hedges & Olkin 1985 asymptotic) mildly under-corrects at n=2 because it assumes a normal sampling distribution; the three-prior sensitivity range bounds that residual uncertainty.

For the underlying interface composition see Part 2; for spatial coherence (join-count statistics) and continuous neighborhood lineage shifts (Family B), see the section after Part 2 ("Two Lenses on Identity").

---

**Bridge to Part 7 (Tier 3, legacy descriptive context only).** The remaining Part 7 is **legacy descriptive material** — scale-dependent cluster counts at 10/20/40μm. It is NOT part of the candidate-finding synthesis. The pre-registration (`temporal_interfaces_plan.md` §2) explicitly selected 10μm a priori and excluded 20/40μm to prevent scale-as-researcher-degree-of-freedom. Reading scale-invariance as redistribution support would reopen exactly that DOF. Part 7 is presented for historical continuity only.


---

# Part 7 — Legacy: Scale-Dependent Cluster Counts at 10/20/40μm

## Status: Legacy Descriptive Context, Not Part of the Synthesis

**Tier label: Tier 3, legacy.** This section pre-dates the temporal interface analysis. It shows cluster counts at multiple SLIC superpixel scales and notes that complexity decreases with coarser scales. We retain it for historical continuity — it documents a structural observation about kidney tissue organization at our chosen scales — but it does NOT bear on the candidate redistribution finding.

**Why we cannot use scale-invariance as evidence here.** The pre-registration `temporal_interfaces_plan.md` §2 explicitly states: *"10μm SLIC superpixels selected a priori as the finest spatial resolution. Results at 20 and 40μm exist as pipeline outputs but are not analyzed in this effort to prevent scale-as-researcher-degree-of-freedom."* Re-introducing 20/40μm as scale-invariance evidence for the candidate finding would reopen exactly that researcher degree of freedom mid-effort. So we don't.


In [ ]:
# Visualize scale-dependent organization for one D7 ROI
d7_roi = df[df['timepoint'] == 'D7']['roi'].unique()[0]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Row 1: Spatial maps at each scale
scales = [10.0, 20.0, 40.0]
scale_labels = ['10um\n(Capillary)', '20um\n(Cellular)', '40um\n(Tubular)']

for idx, (scale, label) in enumerate(zip(scales, scale_labels)):
    ax = axes[0, idx]
    
    # DNA tissue underlay
    render_dna_underlay(ax, d7_roi)
    
    roi_scale_data = df[(df['roi'] == d7_roi) & (df['scale_um'] == scale)]
    
    # Assign colors by cluster
    n_clusters = roi_scale_data['cluster'].nunique()
    colors = plt.cm.tab20(np.linspace(0, 1, n_clusters))
    color_map = {cluster_id: colors[i] for i, cluster_id in enumerate(sorted(roi_scale_data['cluster'].unique()))}
    
    # Plot
    for cluster_id in sorted(roi_scale_data['cluster'].unique()):
        cluster_spx = roi_scale_data[roi_scale_data['cluster'] == cluster_id]
        ax.scatter(cluster_spx['x'], cluster_spx['y'],
                  c=[color_map[cluster_id]], s=20, alpha=0.85,
                  edgecolors='#333333', linewidth=0.2,
                  zorder=2)
    
    ax.set_aspect('equal')
    ax.set_title(f'{label}\n{n_clusters} Spatial Clusters', fontweight='bold', fontsize=12)
    ax.set_xlabel('X (um)')
    ax.set_ylabel('Y (um)')

# Row 2: Cluster characterization at each scale
for idx, scale in enumerate(scales):
    ax = axes[1, idx]
    
    roi_scale_data = df[(df['roi'] == d7_roi) & (df['scale_um'] == scale)]
    
    # Compute cluster profiles
    cluster_profiles = roi_scale_data.groupby('cluster')[markers].mean()
    
    # Assign biological categories based on top marker
    categories = {'Immune': 0, 'Vascular': 0, 'Stromal': 0, 'Activated': 0, 'Mixed': 0}
    
    for cluster_id in cluster_profiles.index:
        profile = cluster_profiles.loc[cluster_id]
        top_marker = profile.idxmax()
        
        if top_marker in ['CD45', 'CD11b', 'Ly6G', 'CD206']:
            categories['Immune'] += 1
        elif top_marker in ['CD31', 'CD34']:
            categories['Vascular'] += 1
        elif top_marker in ['CD140a', 'CD140b']:
            categories['Stromal'] += 1
        elif top_marker == 'CD44':
            categories['Activated'] += 1
        else:
            categories['Mixed'] += 1
    
    # Plot
    category_colors = {
        'Immune': '#E63946',
        'Vascular': '#06AED5',
        'Stromal': '#2A9D8F',
        'Activated': '#F77F00',
        'Mixed': '#95A5A6'
    }
    
    categories_list = [k for k, v in categories.items() if v > 0]
    counts = [categories[k] for k in categories_list]
    colors_bar = [category_colors[k] for k in categories_list]
    
    bars = ax.bar(range(len(categories_list)), counts, color=colors_bar,
                  edgecolor='black', linewidth=1, alpha=0.8)
    
    for i, (cat, count) in enumerate(zip(categories_list, counts)):
        ax.text(i, count + 0.1, str(count), ha='center', fontweight='bold', fontsize=11)
    
    ax.set_xticks(range(len(categories_list)))
    ax.set_xticklabels(categories_list, rotation=45, ha='right', fontsize=10)
    ax.set_ylabel('Number of Clusters', fontweight='bold')
    ax.set_title(f'Cluster Composition\n{len(cluster_profiles)} Total', fontweight='bold', fontsize=12)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim(0, max(counts) * 1.3 if counts else 1)

plt.suptitle(f'Scale-Dependent Tissue Organization: {d7_roi}', 
             fontweight='bold', fontsize=14, y=0.995)
plt.tight_layout()
plt.show()

# Quantify scale transition across all ROIs
print("\n" + "="*80)
print("SCALE-DEPENDENT ORGANIZATION ACROSS ALL ROIS")
print("="*80)

scale_stats = []
for scale in [10.0, 20.0, 40.0]:
    scale_data = df[df['scale_um'] == scale]
    
    clusters_per_roi = scale_data.groupby('roi')['cluster'].nunique()
    
    scale_stats.append({
        'scale': scale,
        'mean_clusters': clusters_per_roi.mean(),
        'std_clusters': clusters_per_roi.std(),
        'min_clusters': clusters_per_roi.min(),
        'max_clusters': clusters_per_roi.max()
    })

print("\nCluster Count by Scale:")
for stat in scale_stats:
    print(f"  {int(stat['scale']):2d}um: {stat['mean_clusters']:.1f} +/- {stat['std_clusters']:.1f} "
          f"(range: {stat['min_clusters']:.0f}-{stat['max_clusters']:.0f})")

# Complexity reduction
complexity_10 = scale_stats[0]['mean_clusters']
complexity_40 = scale_stats[2]['mean_clusters']
reduction = complexity_10 / complexity_40
print(f"\nComplexity Reduction: {reduction:.1f}x from 10um to 40um")
print(f"  -> Fine heterogeneity ({complexity_10:.1f} microenvironments)")
print(f"  -> Coarse structure ({complexity_40:.1f} regional patterns)")

print("\nDESCRIPTIVE NOTES (legacy section, NOT redistribution-hypothesis evidence):")
print("   -> 10um cluster counts are higher than 40um cluster counts (descriptive observation)")
print("   -> Coarser superpixel scales aggregate fine spatial detail into larger zones")
print("   -> Per pre-registration sec 2, the temporal interface analysis uses 10um only;")
print("      20/40um outputs are NOT used as redistribution-hypothesis evidence to avoid")
print("      reopening the scale researcher-degree-of-freedom.")

### Synthesis: What the Pilot Supports

The cluster-count-by-scale plots above are descriptive only and not interpreted as evidence for the candidate finding (per the legacy framing of Part 7 above). The synthesis below pulls from Tier 1 (Parts 2 and 6), the Tier 2 compatibility checks (Part 2.5 spatial coherence; Part 4 cluster concordance), and the Tier 3 negative control (Part 5).

#### Candidate finding (synthesized post-hoc; not pre-registered)

> Stromal-marker-positive tissue area **appears** less stromal-only and more multi-lineage by Sham→D7. We use "appears" deliberately:
> - Family A's CLR transform mathematically couples its two co-movements (stromal-only decrease + triple-positive increase) on the closed simplex.
> - Family C uses related-but-non-identical markers (CD140b vs Family A's CD140a) and different math (raw threshold vs CLR), so its corroboration is convergence across measurement schemes, not replication.
> - Part 2.5 (join-count spatial coherence) provides non-compositional corroboration; Part 4 (Leiden) provides concordance only (same data, different algorithm); Part 5 (neutrophils) argues against — but does NOT rule out — the "uniform-all-cell-types pipeline artifact" alternative; marker- or category-specific normalization artifacts are not addressed.
> - The convergence is **consistent with** redistribution but does not demonstrate it — no lineage tracing, no object-level transition analysis, n=2 mice/timepoint.

#### Alternative hypothesis to discriminate in the follow-up

> The apparent signal could be a **per-ROI sigmoid normalization artifact** rather than biology. Gate 6 sensitivity showed the per-ROI normalization can manufacture apparent compositional shifts: the `endothelial_clr` Sham→D7 finding collapses from g=+2.28 (per-ROI) to g=+0.19 (Sham-reference). The pilot data cannot discriminate redistribution-as-biology from redistribution-as-artifact. A follow-up cohort that uses **Sham-reference threshold normalization upstream in the cell-type annotation engine** (rather than per-ROI sigmoid) would resolve this. n≥10 mice/timepoint; per-animal whole-tissue normalization.

#### Headline n_required ranges (Bayesian shrinkage under three priors)

| Endpoint | Family | n_required (skeptical / neutral / optimistic) |
|---|---|---|
| Triple-overlap fraction Sham→D7 | C | 158 / 17 / 4 |
| Background CD44+ rate Sham→D7 | C | 160 / 18 / 5 |
| Stromal-only CLR Sham→D7 | A | 244 / 34 / 11 |
| CD140b+ within-compartment CD44+ rate | C | 276 / 39 / 14 |
| Triple-positive interface CLR Sham→D7 | A | many hundreds / ~150 / ~50 |

**No prior is designated as default.** Pick the one matching your scepticism about the pilot effects; the range IS the uncertainty statement.

#### Cross-references

- Methods + power discussion + recommendations for the n≥10 follow-up cohort: `notebooks/main_narrative.ipynb` Sections 6 + 7.
- Pre-registration + amendment log: `analysis_plans/temporal_interfaces_plan.md`.
- Throughline architecture for this notebook: `analysis_plans/kidney_notebook_throughline.md`.
- Discretization-vs-continuous trade-offs: `notebooks/methods_validation/01_technical_methods/gradient_discretization.ipynb`.
- I/O integrity benchmark: `notebooks/methods_validation/benchmarks/steinbock_concordance.ipynb`.


In [ ]:
# Compute clusters per ROI at each scale
cluster_summary = df.groupby(['scale_um', 'roi'])['cluster'].nunique().reset_index()
cluster_summary.columns = ['scale_um', 'roi', 'n_clusters']

# Add timepoint metadata
cluster_summary['timepoint'] = cluster_summary['roi'].apply(
    lambda x: 'D1' if 'D1' in x else 'D3' if 'D3' in x else 'D7'
)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Distribution by scale
ax = axes[0]
scales = [10.0, 20.0, 40.0]
positions = [1, 2, 3]
colors = ['#E63946', '#F77F00', '#06AED5']

for scale, pos, color in zip(scales, positions, colors):
    data = cluster_summary[cluster_summary['scale_um'] == scale]['n_clusters']
    parts = ax.violinplot([data], positions=[pos], widths=0.6, 
                          showmeans=True, showextrema=True)
    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_alpha(0.7)

ax.set_xticks(positions)
ax.set_xticklabels([f'{int(s)}μm' for s in scales])
ax.set_ylabel('Number of Spatial Clusters')
ax.set_xlabel('Superpixel Scale')
ax.set_title('Scale-Dependent Tissue Complexity', fontweight='bold', fontsize=12)
ax.grid(axis='y', alpha=0.3)

# Add summary statistics
for scale, pos, color in zip(scales, positions, colors):
    data = cluster_summary[cluster_summary['scale_um'] == scale]['n_clusters']
    mean_val = data.mean()
    ax.text(pos, mean_val + 1, f'{mean_val:.1f}', 
           ha='center', fontweight='bold', color=color)

# Panel B: By timepoint (10μm only)
ax = axes[1]
df_10 = cluster_summary[cluster_summary['scale_um'] == 10.0]
timepoint_order = ['D1', 'D3', 'D7']
tp_colors = {'D1': '#2A9D8F', 'D3': '#F77F00', 'D7': '#E63946'}

for i, tp in enumerate(timepoint_order):
    data = df_10[df_10['timepoint'] == tp]['n_clusters']
    parts = ax.violinplot([data], positions=[i], widths=0.6,
                          showmeans=True, showextrema=True)
    for pc in parts['bodies']:
        pc.set_facecolor(tp_colors[tp])
        pc.set_alpha(0.7)

ax.set_xticks(range(len(timepoint_order)))
ax.set_xticklabels(timepoint_order)
ax.set_ylabel('Number of Clusters (10μm)')
ax.set_xlabel('Timepoint')
ax.set_title('Temporal Stability of Spatial Structure', fontweight='bold', fontsize=12)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary
print("\n" + "="*80)
print("SCALE-DEPENDENT ORGANIZATION")
print("="*80)
for scale in scales:
    data = cluster_summary[cluster_summary['scale_um'] == scale]['n_clusters']
    print(f"\n{int(scale)}μm scale:")
    print(f"  Mean clusters: {data.mean():.1f} ± {data.std():.1f}")
    print(f"  Range: {data.min():.0f}-{data.max():.0f}")

print("\nObservation: cluster count decreases from 10\u03bcm to 40\u03bcm (descriptive; n=2 mice/timepoint, no inferential claim)")
print("   Fine heterogeneity at 10μm → Coarse structure at 40μm")